# WadiDAW Studio: Unified Audio Encoder Selection & Bimodal Ablation Experiment

Satu notebook terpadu yang dapat dijalankan di **Kaggle (GPU T4 / P100)** maupun **Google Colab (GPU T4 / L4 / A100)** untuk seluruh anggota tim (`MEMBER = 1..4`) secara paralel.

---

## Ringkasan Desain Eksperimen (Total 57 Training Run + 2 Inference-Time Removal)

| Fase | Tahap | Deskripsi | Data & Seed | Run Unik Baru |
|---|---|---|---|---:|
| **Phase 1** | **Ablation Awal & Baseline** | 1 Unimodal Text + 4 Unimodal Audio + 4 Bimodal (default `CONFIG`) | Val, seed `42` | 9 |
| **Phase 1** | **Stage 1** (Audio Unimodal OFAT) | OFAT `lr` -> `wd` untuk tiap encoder (`cnn_scratch`, `ast`, `mert`, `beats`) | Val, seed `42` | +12 |
| **Phase 1** | **Stage 2** (Text Unimodal OFAT) | OFAT `lr_text` -> `text_hidden` untuk IndoBERT (`text_only`) | Val, seed `42` | +4 |
| **Phase 1** | **Stage 3** (Bimodal Audio OFAT) | OFAT Audio di mode `bimodal` (Text ikut dilatih dengan default params) | Val, seed `42` | +12 |
| **Phase 2** | **Stage 4** (Bimodal Text OFAT) | OFAT Text pada **Top-2** encoder dari Stage 3 (Audio dikunci di best Stage 3) | Val, seed `42` | +8 |
| **Phase 3** | **Ablation Akhir Test** | 1. Unimodal Audio (best Stage 1)<br>2. Unimodal Text (best Stage 2)<br>3. Bimodal Baseline (encoder Stage 4, default params)<br>4. Bimodal Final (Audio Stage 3 + Text Stage 4) | **Test** (`human` + `synthetic`), **3 seed** (`42, 123, 2026`) | 12 |
| **Phase 3** | **Inference-Time Modality Removal** | 5. Bimodal Text Only (hapus cabang audio dari checkpoint Bimodal Final)<br>6. Bimodal Audio Only (hapus cabang teks dari checkpoint Bimodal Final) | **Test**, 3 seed (tanpa retrain) | **0** |

### Kontrak Ketat Eksperimen
- **Epoch:** **Maksimal 5 epoch** di setiap stage tanpa pengecualian (`MAX_EPOCHS = 5`).
- **Split Holdout Tetap:** 80 lagu train (3.200 sampel) / 20 lagu validation (800 sampel) / 50 lagu test (`test_human` 200 + `test_synthetic` 2.000), *artist-disjoint* dan *prompt-disjoint*.
- **Model Caching Ketat:** Cache key berbasis SHA-256 atas `(protocol_version, encoder, mode, seed, max_epochs, all_hyperparams)` termasuk parameter terkunci (`dropout_audio`, `dropout_text`, `lr_head`, `effective_batch`, `aux_lambda`). Titik baseline tidak pernah dilatih ulang.
- **Dataset Kaggle:**
  - Audio MUSDB18-HQ: `satriamanggala/musdb18hq`
  - Cache waveform 16 kHz: `deathstrawberry/waveform16k`

## Cara Menjalankan Secara Paralel (4 Anggota)

Cukup ubah **`MEMBER`** dan **`PHASE`** pada **Cell 1 (Kontrol Utama)** di bawah ini, lalu jalankan **Run All**.

### Pembagian Kerja Paralel
| `MEMBER` | Phase 1 (`PHASE="phase1"`) | Phase 2 (`PHASE="phase2"`) | Phase 3 (`PHASE="phase3"`) |
|---|---|---|---|
| **1** | `cnn_scratch` (Baseline + Stage 1 + Stage 3) + **`text_only`** (Baseline + Stage 2) | Jalankan jika `cnn_scratch` masuk Top-2 Stage 3 | `unimodal_audio` (best Stage 1, 3 seed) |
| **2** | `ast` (Baseline + Stage 1 + Stage 3) | Jalankan jika `ast` masuk Top-2 Stage 3 | `unimodal_text` (best Stage 2, 3 seed) |
| **3** | `mert` (Baseline + Stage 1 + Stage 3) | Jalankan jika `mert` masuk Top-2 Stage 3 | `bimodal_baseline` (encoder Stage 4, default params, 3 seed) |
| **4** | `beats` (Baseline + Stage 1 + Stage 3) | Jalankan jika `beats` masuk Top-2 Stage 3 | `bimodal_final` (3 seed) + **2 Inference-Time Modality Removal** |

### Alur Sinkronisasi Antar-Fase
1. **Phase 1:** Keempat anggota jalankan `PHASE = "phase1"` secara paralel. Setelah selesai, unduh file ZIP ringan (`wadidaw_results_member<M>_phase1.zip`) dari panel Output Kaggle atau Google Drive.
2. **Gabungkan hasil Phase 1:** Commit folder `outputs/experiment_v3/` ke repo Git (atau unggah ZIP tim sebagai Kaggle Input Dataset / salin ke folder `DRIVE_OUTPUT_DIR` yang sama di Google Drive).
3. **Phase 2:** Dua anggota pemilik **Top-2 encoder Stage 3** menjalankan `PHASE = "phase2"` secara paralel, lalu bagikan hasilnya seperti langkah 2.
4. **Phase 3:** Keempat anggota menjalankan `PHASE = "phase3"` secara paralel (masing-masing hanya 3 run!). Setelah hasil digabungkan, siapa saja dapat menjalankan `PHASE = "report_only"` untuk menghasilkan tabel final CSV dan seluruh grafik analisis.

> **Di Kaggle:** Klik **Add Input** -> tambahkan `deathstrawberry/waveform16k` (wajib) dan `satriamanggala/musdb18hq` (disarankan), lalu aktifkan **GPU (T4/P100)** dan **Internet ON**.
> **Di Colab:** Pilih GPU **L4 / A100 / T4**. Notebook otomatis mengunduh `deathstrawberry/waveform16k` via `kagglehub` (atau membaca dari Google Drive bila sudah tersedia).

In [ ]:
# =============================================================================
# CELL 1: KONTROL UTAMA (UBAH PARAMETER DI SINI SEBELUM RUN ALL)
# =============================================================================
import os

# 1. Identitas anggota (1..4) -> menentukan shard encoder & tugas paralel
MEMBER = int(os.environ.get("WADIDAW_MEMBER", "1"))  # 1=cnn_scratch+text, 2=ast, 3=mert, 4=beats

# 2. Fase yang dijalankan:
#    "phase1"      -> Ablation Awal & Baseline + Stage 1 + Stage 2 (M1) + Stage 3
#    "phase2"      -> Stage 4 (Bimodal Text OFAT pada Top-2 Stage 3)
#    "phase3"      -> Ablation Akhir Test (3 seed, paralel 1 tugas/member) + Inference-Time Removal
#    "auto"        -> Jalankan semua fase yang prasyaratnya sudah siap untuk MEMBER ini
#    "report_only" -> Tidak training; hanya agregasi tabel CSV, grafik, parity check, dan ZIP
PHASE = os.environ.get("WADIDAW_PHASE", "phase1")

# 3. Opsi kontrol tambahan (biasanya biarkan default)
FORCE_RERUN = False                 # True = abaikan Model Cache & latih ulang
STAGE4_ENCODERS = None              # None = jalankan MY_ENCODER bila masuk Top-2; atau ["all_top2"]
PHASE3_TASKS = None                 # None = otomatis 1 tugas/member; atau ["all"] / ["bimodal_final"]
ALSO_TEST_STAGE4_ENC_AUDIO_ONLY = False  # True = jika pemenang Stage 1 != Stage 4, test juga audio_only encoder Stage 4
TIME_BUDGET_MIN = 660               # Batas aman sesi (660 mnt = 11 jam, cocok untuk limit 12 jam Kaggle)
EXPORT_FINAL_CHECKPOINTS = True     # True = ikut salin/zip checkpoint .pt final (ckpt tetap dibuat untuk Inference Removal)

# 4. Sumber Repository & Dataset Kaggle
REPO_URL = "https://github.com/savitar393/wadidaw-studio.git"
REPO_BRANCH = "feat/notebook-new"   # fallback otomatis ke 'main' jika branch tidak ditemukan
KAGGLE_MUSDB_SLUG = "satriamanggala/musdb18hq"
KAGGLE_WAVEFORM_SLUG = "deathstrawberry/waveform16k"

# 5. Opsi Google Drive (hanya berlaku di Colab; diabaikan otomatis di Kaggle/Lokal)
USE_GOOGLE_DRIVE = True
DRIVE_PROJECT_SUBDIR = "ml_dabd_nlp/ml"
DRIVE_OUTPUT_SUBDIR = "2.Dokumentasi/experiment_v3"  # folder bersama di Drive agar auto-sync antar member
AUTO_TRIGGER_COLAB_DOWNLOAD = False  # True = otomatis trigger browser download ZIP di akhir Colab

# 6. Opsi dataset mentah MUSDB18-HQ saat waveform16k sudah lengkap 1.500 file:
#    False = tidak perlu unduh 31 GB WAV di Colab jika cache waveform16k sudah 100% lengkap
DOWNLOAD_MUSDB_IF_WAVEFORM_COMPLETE = False

# 7. Direktori tambahan berisi hasil JSON/ZIP tim (opsional, untuk merge manual)
EXTRA_RESULTS_PATHS = []

assert MEMBER in (1, 2, 3, 4), f"MEMBER harus 1, 2, 3, atau 4 (bukan {MEMBER})"
assert PHASE in ("phase1", "phase2", "phase3", "auto", "report_only"), f"PHASE tidak valid: {PHASE}"
print(f"Konfigurasi aktif -> MEMBER={MEMBER} | PHASE={PHASE} | FORCE_RERUN={FORCE_RERUN}")

In [ ]:
# =============================================================================
# CELL 2: SETUP LINGKUNGAN (OTOMATIS KAGGLE / COLAB / LOKAL + GPU)
# =============================================================================
import importlib
import json
import shutil
import socket
import subprocess
import sys
import zipfile
from pathlib import Path


def detect_environment() -> str:
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") or Path("/kaggle/working").exists():
        return "kaggle"
    try:
        import google.colab  # noqa: F401
        return "colab"
    except ImportError:
        pass
    if Path("/content").exists() and not (Path.cwd() / "wadidaw").is_dir():
        return "colab"
    return "local"


ENV_NAME = detect_environment()
IN_KAGGLE = ENV_NAME == "kaggle"
IN_COLAB = ENV_NAME == "colab"
print(f"Terdeteksi environment: {ENV_NAME.upper()}")


def run_cmd(cmd: str, check: bool = True) -> subprocess.CompletedProcess:
    print("+", cmd)
    return subprocess.run(cmd, shell=True, check=check)


def has_internet(host: str = "github.com") -> bool:
    try:
        socket.gethostbyname(host)
        return True
    except OSError:
        return False


# --- Mount Google Drive jika di Colab ----------------------------------------
DRIVE_ROOT = Path("/content/drive/MyDrive")
DRIVE_PROJECT_DIR = DRIVE_ROOT / DRIVE_PROJECT_SUBDIR
DRIVE_CACHE_ROOT = DRIVE_PROJECT_DIR / "1.Dataset" / "cache"
DRIVE_OUTPUT_DIR = DRIVE_PROJECT_DIR / DRIVE_OUTPUT_SUBDIR
DRIVE_AVAILABLE = False

if IN_COLAB and USE_GOOGLE_DRIVE:
    if not DRIVE_ROOT.exists():
        from google.colab import drive
        drive.mount("/content/drive")
    DRIVE_AVAILABLE = DRIVE_ROOT.exists()
    print("Google Drive tersedia:", DRIVE_AVAILABLE, "| output:", DRIVE_OUTPUT_DIR)

# --- Tentukan & siapkan direktori repo ---------------------------------------
if IN_KAGGLE:
    WORK_ROOT = Path("/kaggle/working")
    INPUT_ROOT = Path("/kaggle/input")
    REPO_DIR = WORK_ROOT / "wadidaw-studio"
elif IN_COLAB:
    WORK_ROOT = Path("/content")
    INPUT_ROOT = Path("/content/kaggle_inputs")
    REPO_DIR = WORK_ROOT / "wadidaw-studio"
else:
    WORK_ROOT = Path.cwd()
    INPUT_ROOT = Path("/kaggle/input")
    REPO_DIR = Path.cwd()
    for cand in (Path.cwd(), *Path.cwd().parents):
        if (cand / "wadidaw").is_dir():
            REPO_DIR = cand
            break


def find_uploaded_repo() -> Path | None:
    if not INPUT_ROOT.exists():
        return None
    for pat in ("*", "*/*", "*/*/*"):
        for cand in INPUT_ROOT.glob(pat):
            try:
                if (cand / "wadidaw").is_dir() and (cand / "data" / "splits").is_dir():
                    return cand
            except OSError:
                continue
    return None


if not (REPO_DIR / "wadidaw").is_dir():
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR, ignore_errors=True)
    if has_internet("github.com"):
        res = run_cmd(f"git clone --depth 1 --branch {REPO_BRANCH} {REPO_URL} {REPO_DIR}", check=False)
        if res.returncode != 0:
            print(f"Branch '{REPO_BRANCH}' tidak ditemukan -> fallback ke default branch.")
            run_cmd(f"git clone --depth 1 {REPO_URL} {REPO_DIR}", check=True)
    else:
        uploaded = find_uploaded_repo()
        if uploaded is not None:
            print("Internet OFF -> salin repo dari input:", uploaded)
            shutil.copytree(uploaded, REPO_DIR)
        else:
            raise RuntimeError(
                "Tidak bisa clone repo (Internet OFF). Aktifkan Internet ON di Kaggle/Colab "
                "atau unggah repo sebagai Kaggle Input Dataset."
            )
else:
    print("Repo sudah siap:", REPO_DIR)

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

# --- Install dependensi yang dibutuhkan (idempotent) -------------------------
if IN_KAGGLE or IN_COLAB:
    if has_internet("pypi.org"):
        pkgs = ["'transformers>=4.40'", "soundfile", "scipy", "kagglehub", "matplotlib", "pandas"]
        run_cmd(f"{sys.executable} -m pip install -q {' '.join(pkgs)}", check=False)
        run_cmd(f"{sys.executable} -m pip install -q --no-deps nnAudio", check=False)
    else:
        print("Peringatan: Internet OFF, melewati pip install.")

import torch

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision("high")
    props = torch.cuda.get_device_properties(0)
    GPU_NAME = props.name
    GPU_VRAM_GB = props.total_memory / 1e9
    BF16_SUPPORTED = bool(torch.cuda.is_bf16_supported())
    print(f"PyTorch {torch.__version__} | GPU: {GPU_NAME} ({GPU_VRAM_GB:.1f} GB) | bf16={BF16_SUPPORTED}")
else:
    GPU_NAME = "CPU"
    GPU_VRAM_GB = 0.0
    BF16_SUPPORTED = False
    print(f"PyTorch {torch.__version__} | PERINGATAN: CUDA tidak aktif, berjalan di CPU!")

In [ ]:
# =============================================================================
# CELL 3: DETEKSI & ATTACH DATASET KAGGLE (waveform16k + musdb18hq)
# =============================================================================
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import soundfile as sf

from wadidaw.data.dataset import SAMPLES_ROOT
from wadidaw.data.features import extract_logmel_stems, resample_to_16k_mono
from wadidaw.data.prompts import STEM_NAMES
import wadidaw.data.musdb as _musdb_module

WAVEFORM_CACHE_DIR = REPO_DIR / "data" / "cache" / "waveform16k"
LOGMEL_CACHE_DIR = REPO_DIR / "data" / "cache" / "logmel"
STEM_MEL_POWER_FILE = REPO_DIR / "data" / "cache" / "stem_mel_power_16k.npz"
EXPECTED_EXCERPTS = 1500  # 800 train + 200 val + 500 test


def count_npy(path: Path) -> int:
    if not path.exists():
        return 0
    return sum(1 for _ in path.rglob("*.npy"))


def _first_npy_shape(path: Path) -> tuple[int, ...] | None:
    for split in ("train", "validation", "test"):
        d = path / split
        if d.is_dir():
            first = next(d.glob("*.npy"), None)
            if first is not None:
                try:
                    return tuple(np.load(first, mmap_mode="r").shape)
                except Exception:
                    return None
    # Juga dukung struktur flat (train_*.npy langsung di root dataset)
    first_flat = next(path.glob("train_*.npy"), None)
    if first_flat is not None:
        try:
            return tuple(np.load(first_flat, mmap_mode="r").shape)
        except Exception:
            return None
    return None


def is_waveform_cache_dir(path: Path) -> bool:
    has_structured = (path / "train").is_dir()
    has_flat = next(path.glob("train_*.npy"), None) is not None
    if not (has_structured or has_flat):
        return False
    shape = _first_npy_shape(path)
    return shape is not None and len(shape) == 2 and shape[0] == 4 and shape[1] >= 16000


def is_logmel_cache_dir(path: Path) -> bool:
    if not (path / "train").is_dir():
        return False
    shape = _first_npy_shape(path)
    return shape == (4, 64, 800)


def looks_like_musdb(path: Path) -> bool:
    try:
        return (
            (path / "train").is_dir()
            and (path / "test").is_dir()
            and next((path / "train").glob("*/mixture.wav"), None) is not None
        )
    except OSError:
        return False


def search_dir_tree(roots: list[Path], predicate) -> Path | None:
    for root in roots:
        if not root.exists():
            continue
        if predicate(root):
            return root
        for pat in ("*", "*/*", "*/*/*", "*/*/*/*"):
            for cand in sorted(root.glob(pat)):
                if cand.is_dir() and predicate(cand):
                    return cand
    return None


def link_or_copy_cache(source: Path, target: Path, copy_to_local: bool, label: str) -> None:
    if source.resolve() == target.resolve():
        print(f"[{label}] sudah berada di target: {target} ({count_npy(target)} file)")
        return
    if target.is_symlink():
        if target.resolve() == source.resolve():
            print(f"[{label}] symlink sudah benar: {target} -> {source}")
            return
        target.unlink()
    elif target.exists():
        if count_npy(target) >= count_npy(source) and count_npy(target) > 0:
            print(f"[{label}] cache lokal sudah lengkap: {target} ({count_npy(target)} file)")
            return
        if not any(target.iterdir()):
            target.rmdir()

    target.parent.mkdir(parents=True, exist_ok=True)
    if copy_to_local:
        print(f"[{label}] menyalin {count_npy(source)} file ke SSD lokal ({target})...")
        shutil.copytree(source, target, dirs_exist_ok=True)
        print(f"[{label}] selesai copy ke SSD lokal ({count_npy(target)} file).")
    else:
        target.symlink_to(source, target_is_directory=True)
        print(f"[{label}] symlink: {target} -> {source} ({count_npy(source)} file)")


def download_kaggle_dataset(slug: str) -> Path | None:
    if not has_internet("kaggle.com"):
        print(f"[kagglehub] Internet OFF; tidak bisa mengunduh {slug}")
        return None
    try:
        import kagglehub
        print(f"[kagglehub] mengunduh dataset Kaggle: {slug} ...")
        dl_path = Path(kagglehub.dataset_download(slug))
        print(f"[kagglehub] selesai: {dl_path}")
        return dl_path
    except Exception as exc:
        print(f"[kagglehub] gagal mengunduh {slug}: {exc}")
        return None


# --- 1. Resolve cache waveform16k (prioritas: lokal -> Kaggle Input -> Drive -> kagglehub) ---
wf_roots = [WAVEFORM_CACHE_DIR]
if INPUT_ROOT.exists():
    wf_roots.append(INPUT_ROOT)
if DRIVE_AVAILABLE:
    wf_roots.extend([DRIVE_CACHE_ROOT, DRIVE_PROJECT_DIR / "1.Dataset"])

found_wf = search_dir_tree(wf_roots, is_waveform_cache_dir)
if found_wf is None and INPUT_ROOT.exists():
    # Fallback jika dataset waveform16k di-upload dalam bentuk .zip
    wf_zips = [z for z in sorted(INPUT_ROOT.glob("**/*.zip")) if "waveform" in z.name.lower()]
    if wf_zips:
        extract_dst = WORK_ROOT / "_extracted_waveform16k"
        extract_dst.mkdir(parents=True, exist_ok=True)
        print(f"[waveform16k] mengekstrak {wf_zips[0]} -> {extract_dst} ...")
        with zipfile.ZipFile(wf_zips[0], "r") as zf:
            zf.extractall(extract_dst)
        found_wf = search_dir_tree([extract_dst], is_waveform_cache_dir)
if found_wf is None:
    dl_wf = download_kaggle_dataset(KAGGLE_WAVEFORM_SLUG)
    if dl_wf is not None:
        found_wf = search_dir_tree([dl_wf], is_waveform_cache_dir)

if found_wf is not None:
    # Di Colab dari Drive (FUSE lambat) -> copy ke SSD lokal; di Kaggle/local/kagglehub SSD -> symlink.
    from_drive = DRIVE_AVAILABLE and str(found_wf).startswith(str(DRIVE_ROOT))
    link_or_copy_cache(found_wf, WAVEFORM_CACHE_DIR, copy_to_local=from_drive, label="waveform16k")

wf_count = count_npy(WAVEFORM_CACHE_DIR)
print(f"Status cache waveform16k: {wf_count}/{EXPECTED_EXCERPTS} file di {WAVEFORM_CACHE_DIR}")

# --- 2. Resolve dataset MUSDB18-HQ (attach bila ada; unduh hanya jika dibutuhkan) ---
musdb_roots = [REPO_DIR / "data" / "raw" / "musdb18hq"]
if INPUT_ROOT.exists():
    musdb_roots.append(INPUT_ROOT)
if DRIVE_AVAILABLE:
    musdb_roots.extend([DRIVE_PROJECT_DIR / "1.Dataset" / "dataset", DRIVE_PROJECT_DIR / "1.Dataset"])

MUSDB_ROOT_FOUND = search_dir_tree(musdb_roots, looks_like_musdb)
if MUSDB_ROOT_FOUND is None and INPUT_ROOT.exists() and (wf_count < EXPECTED_EXCERPTS or DOWNLOAD_MUSDB_IF_WAVEFORM_COMPLETE):
    musdb_zips = [z for z in sorted(INPUT_ROOT.glob("**/*.zip")) if "musdb" in z.name.lower()]
    if musdb_zips:
        extract_dst = Path("/kaggle/tmp/musdb18hq" if IN_KAGGLE else WORK_ROOT / "_extracted_musdb18hq")
        extract_dst.mkdir(parents=True, exist_ok=True)
        print(f"[musdb18hq] mengekstrak {musdb_zips[0]} -> {extract_dst} ...")
        with zipfile.ZipFile(musdb_zips[0], "r") as zf:
            zf.extractall(extract_dst)
        MUSDB_ROOT_FOUND = search_dir_tree([extract_dst], looks_like_musdb)
if MUSDB_ROOT_FOUND is None and (wf_count < EXPECTED_EXCERPTS or DOWNLOAD_MUSDB_IF_WAVEFORM_COMPLETE):
    dl_musdb = download_kaggle_dataset(KAGGLE_MUSDB_SLUG)
    if dl_musdb is not None:
        MUSDB_ROOT_FOUND = search_dir_tree([dl_musdb], looks_like_musdb)

if MUSDB_ROOT_FOUND is not None:
    _musdb_module.MUSDB_ROOT = MUSDB_ROOT_FOUND
    raw_link = REPO_DIR / "data" / "raw" / "musdb18hq"
    raw_link.parent.mkdir(parents=True, exist_ok=True)
    if not raw_link.exists() and not raw_link.is_symlink():
        raw_link.symlink_to(MUSDB_ROOT_FOUND, target_is_directory=True)
    split_manifest = json.loads((REPO_DIR / "data" / "splits" / "musdb18hq_v1.json").read_text())
    songs = [m for sp in split_manifest["splits"].values() for m in sp]
    missing_songs = [m["song"] for m in songs if not (MUSDB_ROOT_FOUND / str(m["relative_dir"])).is_dir()]
    print(f"MUSDB18-HQ terpasang: {MUSDB_ROOT_FOUND} | songs={len(songs)}, missing={len(missing_songs)}")
else:
    print("MUSDB18-HQ mentah tidak terpasang (aman jika waveform16k sudah lengkap 1.500 file).")

# --- 3. Fallback bangun waveform16k yang hilang bila < 1.500 file ------------
ALL_MANIFESTS = [
    SAMPLES_ROOT / "train_samples.json",
    SAMPLES_ROOT / "validation_samples.json",
    SAMPLES_ROOT / "test_human_samples.json",
    SAMPLES_ROOT / "test_synthetic_samples.json",
]


def load_manifest_records(path: Path) -> list[dict]:
    data = json.loads(Path(path).read_text())
    return list(data["samples"]) if isinstance(data, dict) and "samples" in data else list(data)


def excerpt_id_and_split(audio_path: str) -> tuple[str, str]:
    p = Path(audio_path)
    return p.stem, p.parent.name


def waveform_path_for(audio_path: str) -> Path:
    ex_id, split_tag = excerpt_id_and_split(audio_path)
    structured = WAVEFORM_CACHE_DIR / split_tag / f"{ex_id}.npy"
    if structured.is_file():
        return structured
    flat = WAVEFORM_CACHE_DIR / f"{ex_id}.npy"
    if flat.is_file():
        return flat
    return structured


unique_excerpts: dict[str, dict] = {}
for mpath in ALL_MANIFESTS:
    for rec in load_manifest_records(mpath):
        unique_excerpts.setdefault(str(rec["audio_path"]), rec)

missing_wf = [ap for ap in unique_excerpts if not waveform_path_for(ap).is_file()]
if missing_wf:
    if MUSDB_ROOT_FOUND is None:
        raise FileNotFoundError(
            f"Ada {len(missing_wf)} file waveform16k yang hilang dan MUSDB18-HQ tidak ditemukan. "
            f"Tambahkan dataset Kaggle '{KAGGLE_WAVEFORM_SLUG}' (dan '{KAGGLE_MUSDB_SLUG}') via Add Input."
        )
    print(f"Membangun {len(missing_wf)} excerpt waveform16k yang kurang dari MUSDB18-HQ...")

    def _build_missing_wf(ap: str) -> int:
        rec = unique_excerpts[ap]
        out_p = waveform_path_for(ap)
        _, split_tag = excerpt_id_and_split(ap)
        subset = "train" if split_tag in ("train", "validation") else "test"
        song_dir = MUSDB_ROOT_FOUND / subset / str(rec["song"])
        start = int(round(float(rec["excerpt_start_sec"]) * 44100))
        frames = int(round(float(rec["duration_sec"]) * 44100))
        stems = []
        for s_name in STEM_NAMES:
            wav, sr = sf.read(song_dir / f"{s_name}.wav", start=start, frames=frames, dtype="float32")
            stems.append(resample_to_16k_mono(wav, orig_sr=sr))
        out_p.parent.mkdir(parents=True, exist_ok=True)
        np.save(out_p, np.stack(stems, axis=0).astype(np.float16))
        return 1

    with ThreadPoolExecutor(max_workers=min(8, os.cpu_count() or 2)) as pool:
        built = sum(pool.map(_build_missing_wf, missing_wf))
    print(f"Selesai membangun {built} waveform16k.")

assert all(waveform_path_for(ap).is_file() for ap in unique_excerpts), "Cache waveform16k belum lengkap!"
print(f"Verifikasi waveform16k OK: {len(unique_excerpts)}/1500 excerpt lengkap.")

# --- 4. Cek log-mel cache (opsional) & bangun cache ringan stem_mel_power (1.5 MB) ---
lm_roots = [LOGMEL_CACHE_DIR]
if INPUT_ROOT.exists():
    lm_roots.append(INPUT_ROOT)
if DRIVE_AVAILABLE:
    lm_roots.extend([DRIVE_CACHE_ROOT, DRIVE_PROJECT_DIR / "1.Dataset"])
found_lm = search_dir_tree(lm_roots, is_logmel_cache_dir)
if found_lm is not None and found_lm.resolve() != LOGMEL_CACHE_DIR.resolve():
    from_drive = DRIVE_AVAILABLE and str(found_lm).startswith(str(DRIVE_ROOT))
    link_or_copy_cache(found_lm, LOGMEL_CACHE_DIR, copy_to_local=from_drive, label="logmel")


def build_or_load_stem_mel_power() -> dict[str, np.ndarray]:
    """Muat/bangun `(4, 64)` mean mel-power untuk metrik spektral `evaluate_model`.

    Tidak butuh dataset logmel 593 MB terpisah: jika cache logmel tidak ada,
    dihitung langsung dari `waveform16k` dalam ~15-25 detik lalu di-cache (~1.5 MB).
    """
    if STEM_MEL_POWER_FILE.is_file():
        loaded = np.load(STEM_MEL_POWER_FILE)
        if all(ap in loaded.files for ap in unique_excerpts):
            print(f"Memuat stem_mel_power cache: {STEM_MEL_POWER_FILE} ({len(loaded.files)} excerpt)")
            return {k: loaded[k].astype(np.float32) for k in loaded.files}

    use_logmel = all((REPO_DIR / ap).is_file() for ap in unique_excerpts)
    src_desc = "log-mel cache" if use_logmel else "waveform16k (extract_logmel_stems)"
    print(f"Membangun stem_mel_power (1.500 excerpt) dari {src_desc}...")

    def _compute_one(ap: str) -> tuple[str, np.ndarray]:
        if use_logmel:
            lm_db = np.load(REPO_DIR / ap).astype(np.float32)
        else:
            wf = np.load(waveform_path_for(ap)).astype(np.float32)
            lm_db = extract_logmel_stems(wf)
        mel_pow = np.power(10.0, np.clip(lm_db, -100.0, 20.0) / 10.0).mean(axis=-1).astype(np.float32)
        return ap, mel_pow

    out: dict[str, np.ndarray] = {}
    with ThreadPoolExecutor(max_workers=min(8, os.cpu_count() or 2)) as pool:
        for ap, mp in pool.map(_compute_one, list(unique_excerpts)):
            out[ap] = mp

    STEM_MEL_POWER_FILE.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(STEM_MEL_POWER_FILE, **out)
    print(f"Tersimpan {STEM_MEL_POWER_FILE} ({STEM_MEL_POWER_FILE.stat().st_size / 1e6:.2f} MB)")
    return out


RAW_STEM_MEL_POWER = build_or_load_stem_mel_power()

In [ ]:
# =============================================================================
# CELL 4: KONFIGURASI PROTOKOL, SEARCH SPACE, & STRICT MODEL CACHING
# =============================================================================
import copy
import hashlib
import random
import time

from wadidaw.data.features import ACTIVE_THRESHOLD_DB
from wadidaw.models.structured import compute_multimodal_loss
from wadidaw.training.evaluate import evaluate_model


def system_ram_gb() -> float:
    try:
        return (os.sysconf("SC_PHYS_PAGES") * os.sysconf("SC_PAGE_SIZE")) / 1e9
    except (ValueError, AttributeError):
        return 0.0


PROTOCOL_VERSION = "v3_ofat4stage_ep5_strict"
MAX_EPOCHS = 5  # Max 5 epoch untuk setiap stage, tanpa pengecualian!
EFFECTIVE_BATCH = 16

CONFIG = {
    "train_manifest": str(SAMPLES_ROOT / "train_samples.json"),
    "val_manifest": str(SAMPLES_ROOT / "validation_samples.json"),
    "test_human_manifest": str(SAMPLES_ROOT / "test_human_samples.json"),
    "test_synthetic_manifest": str(SAMPLES_ROOT / "test_synthetic_samples.json"),
    "text_model": "indobenchmark/indobert-base-p1",
    "d_model": 128,
    "text_hidden": 128,
    "dropout_text": 0.15,
    "dropout_audio": 0.10,
    "batch_size": 4,
    "grad_accum_steps": 4,
    "epochs": MAX_EPOCHS,
    "patience": 3,
    "lr_head": 1e-3,
    "lr_audio": 1e-4,
    "lr_scratch": 1e-3,
    "lr_text": 1e-4,
    "wd_audio": 1e-4,
    "wd_text": 1e-4,
    "wd_head": 1e-4,
    "eta_min": 1e-6,
    "grad_clip": 1.0,
    "tuning_seed": 42,
    "final_seeds": [42, 123, 2026],
    "use_amp": True,
    "encoders": ["cnn_scratch", "ast", "mert", "beats"],
    "modes": ["bimodal", "audio_only", "text_only"],
    "waveform_rate": 16000,
}

# Bobot auxiliary loss per mode (lambda_loudness, lambda_balance)
AUX_LAMBDA = {
    "bimodal": (0.1, 0.1),
    "audio_only": (0.1, 0.0),
    "text_only": (0.0, 0.1),
}

# Search space sesuai desain eksperimen (semua nilai baseline CONFIG ada di dalam search space)
AUDIO_SEARCH_SPACE = {
    "pretrained": {
        "lr_audio": [5e-5, 1e-4, 2e-4],
        "wd_audio": [1e-4, 1e-3],
    },
    "scratch": {
        "lr_scratch": [5e-4, 1e-3, 2e-3],
        "wd_audio": [1e-4, 1e-3],
    },
}

TEXT_SEARCH_SPACE = {
    "lr_text": [5e-5, 1e-4, 2e-4],
    "text_hidden": [64, 128, 256],
}

# Pembagian paralel 4 anggota:
# Member 1 (cnn_scratch paling cepat) juga menangani text_only agar beban tim seimbang.
SHARDS = {
    1: {"encoder": "cnn_scratch", "owns_text_unimodal": True,  "phase3_task": "unimodal_audio"},
    2: {"encoder": "ast",         "owns_text_unimodal": False, "phase3_task": "unimodal_text"},
    3: {"encoder": "mert",        "owns_text_unimodal": False, "phase3_task": "bimodal_baseline"},
    4: {"encoder": "beats",       "owns_text_unimodal": False, "phase3_task": "bimodal_final"},
}

MY_ENCODER = SHARDS[MEMBER]["encoder"]
OWNS_TEXT_UNIMODAL = SHARDS[MEMBER]["owns_text_unimodal"]

# Adaptasi physical batch size sesuai VRAM GPU (effective_batch tetap terkunci = 16)
def preferred_batch_config(encoder_name: str, mode: str) -> tuple[int, int]:
    # cnn_scratch memakai BatchNorm1d -> tetap 4x4 agar statistik batch identik di semua GPU.
    if mode != "text_only" and encoder_name == "cnn_scratch":
        return 4, 4
    if torch.cuda.is_available():
        if GPU_VRAM_GB >= 36.0:   # A100 40/80 GB
            return 16, 1
        if GPU_VRAM_GB >= 20.0:   # L4 24 GB
            return 8, 2
    return 4, 4


NUM_WORKERS = min(4, os.cpu_count() or 2) if torch.cuda.is_available() else 0
PIN_MEMORY = bool(torch.cuda.is_available())
PRELOAD_WAVEFORM_IN_RAM = system_ram_gb() >= 22.0
AMP_DTYPE = torch.bfloat16 if (torch.cuda.is_available() and BF16_SUPPORTED) else torch.float16
# BEATs tidak stabil dengan AMP FP16 (T4/P100); aman dengan BF16 (L4/A100)
AMP_DISABLED_ENCODERS = {"beats"} if AMP_DTYPE == torch.float16 else set()
SESSION_START_TIME = time.time()

# --- Direktori output terstruktur --------------------------------------------
OUT_DIR = REPO_DIR / "outputs" / "experiment_v3"
CACHE_DIR = OUT_DIR / "run_cache"
BASELINE_DIR = OUT_DIR / "baseline"
STAGES_DIR = OUT_DIR / "stages"
TEST_DIR = OUT_DIR / "test_ablation"
CKPT_DIR = OUT_DIR / "checkpoints"
REPORT_DIR = OUT_DIR / "reports"
FIG_DIR = OUT_DIR / "figures"

for _d in (OUT_DIR, CACHE_DIR, BASELINE_DIR, STAGES_DIR, TEST_DIR, CKPT_DIR, REPORT_DIR, FIG_DIR):
    _d.mkdir(parents=True, exist_ok=True)


# --- Auto-merge hasil tim dari Drive / Kaggle Input / EXTRA_RESULTS_PATHS ----
def _merge_tree(src: Path, dst: Path) -> int:
    copied = 0
    for item in src.rglob("*"):
        if not item.is_file():
            continue
        rel = item.relative_to(src)
        target = dst / rel
        if target.exists() and target.stat().st_size == item.stat().st_size:
            continue
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(item, target)
        copied += 1
    return copied


def restore_and_merge_team_outputs() -> None:
    sources: list[Path] = []
    if DRIVE_AVAILABLE and DRIVE_OUTPUT_DIR.is_dir():
        sources.append(DRIVE_OUTPUT_DIR)
    for extra in EXTRA_RESULTS_PATHS:
        p = Path(extra)
        if p.exists():
            sources.append(p)
    # Cek apakah ada hasil experiment_v3 atau ZIP hasil tim yang di-attach di /kaggle/input
    if INPUT_ROOT.exists():
        for pat in ("*", "*/*", "*/*/*"):
            for cand in INPUT_ROOT.glob(pat):
                if cand.is_dir() and any((cand / sub).is_dir() for sub in ("run_cache", "stages", "baseline", "test_ablation")):
                    sources.append(cand)
                elif cand.is_file() and cand.suffix == ".zip" and "wadidaw_results" in cand.name:
                    sources.append(cand)

    total_copied = 0
    for src in sources:
        if src.is_dir():
            n = _merge_tree(src, OUT_DIR)
            if n:
                print(f"[merge] +{n} file dari folder: {src}")
            total_copied += n
        elif src.is_file() and src.suffix == ".zip":
            tmp_dir = WORK_ROOT / "_tmp_merge" / src.stem
            tmp_dir.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(src, "r") as zf:
                zf.extractall(tmp_dir)
            # Deteksi apakah isi zip langsung run_cache/ atau dibungkus experiment_v3/
            root_inside = tmp_dir / "experiment_v3" if (tmp_dir / "experiment_v3").is_dir() else tmp_dir
            n = _merge_tree(root_inside, OUT_DIR)
            shutil.rmtree(tmp_dir, ignore_errors=True)
            if n:
                print(f"[merge] +{n} file dari ZIP: {src.name}")
            total_copied += n
    if total_copied == 0:
        print("[merge] tidak ada file hasil tambahan baru untuk digabungkan.")


restore_and_merge_team_outputs()


def sync_to_drive(include_checkpoints: bool = False) -> None:
    if not DRIVE_AVAILABLE:
        return
    DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    for sub in ("run_cache", "baseline", "stages", "test_ablation", "reports", "figures"):
        src = OUT_DIR / sub
        if src.is_dir() and any(src.iterdir()):
            _merge_tree(src, DRIVE_OUTPUT_DIR / sub)
    if include_checkpoints and CKPT_DIR.is_dir() and any(CKPT_DIR.iterdir()):
        _merge_tree(CKPT_DIR, DRIVE_OUTPUT_DIR / "checkpoints")


# --- Strict Model Caching (Complete Canonical Key) ---------------------------
def make_config(overrides: dict | None = None, encoder_name: str | None = None, mode: str = "bimodal") -> dict:
    cfg = copy.deepcopy(CONFIG)
    if overrides:
        cfg.update(overrides)
    if encoder_name is not None:
        bs, accum = preferred_batch_config(encoder_name, mode)
        cfg["batch_size"] = bs
        cfg["grad_accum_steps"] = accum
    cfg["epochs"] = MAX_EPOCHS  # Paksa 5 epoch tanpa pengecualian
    assert int(cfg["batch_size"]) * int(cfg["grad_accum_steps"]) == EFFECTIVE_BATCH
    return cfg


def canonical_cache_payload(encoder_name: str, mode: str, seed: int, cfg: dict) -> dict:
    """Bangun kunci cache lengkap (protocol_version, encoder, mode, seed, max_epochs, all_hyperparams).

    Menyertakan seluruh parameter terkunci (dropout_audio, dropout_text, lr_head,
    effective_batch, d_model, aux_lambda) agar bebas tabrakan lintas mode/seed/protokol.
    """
    is_scratch = (encoder_name == "cnn_scratch")
    eff_encoder = "none" if mode == "text_only" else str(encoder_name)
    all_hparams = {
        "lr_audio": None if mode == "text_only" else float(cfg["lr_scratch"] if is_scratch else cfg["lr_audio"]),
        "wd_audio": None if mode == "text_only" else float(cfg["wd_audio"]),
        "dropout_audio": None if mode == "text_only" else float(cfg["dropout_audio"]),
        "lr_text": None if mode == "audio_only" else float(cfg["lr_text"]),
        "wd_text": None if mode == "audio_only" else float(cfg["wd_text"]),
        "dropout_text": None if mode == "audio_only" else float(cfg["dropout_text"]),
        "text_hidden": None if mode == "audio_only" else int(cfg["text_hidden"]),
        "text_model": None if mode == "audio_only" else str(cfg["text_model"]),
        "d_model": int(cfg["d_model"]),
        "lr_head": float(cfg["lr_head"]),
        "wd_head": float(cfg["wd_head"]),
        "eta_min": float(cfg["eta_min"]),
        "grad_clip": float(cfg["grad_clip"]),
        "effective_batch": int(cfg["batch_size"]) * int(cfg["grad_accum_steps"]),
        "aux_lambda": [float(x) for x in AUX_LAMBDA[mode]],
    }
    return {
        "protocol_version": PROTOCOL_VERSION,
        "encoder": eff_encoder,
        "mode": str(mode),
        "seed": int(seed),
        "max_epochs": int(cfg["epochs"]),
        "patience": int(cfg["patience"]),
        "all_hyperparams": all_hparams,
    }


def cache_key_hash(payload: dict) -> str:
    serialized = json.dumps(payload, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(serialized.encode("utf-8")).hexdigest()[:16]


def cache_file_path(payload: dict) -> Path:
    h = cache_key_hash(payload)
    return CACHE_DIR / f"{payload['encoder']}__{payload['mode']}__s{payload['seed']}__ep{payload['max_epochs']}__{h}.json"


def lookup_run_cache(
    encoder_name: str,
    mode: str,
    seed: int,
    cfg: dict,
    require_test: bool = False,
    require_checkpoint: bool = False,
) -> dict | None:
    if FORCE_RERUN:
        return None
    payload = canonical_cache_payload(encoder_name, mode, seed, cfg)
    cpath = cache_file_path(payload)
    if not cpath.is_file():
        return None
    try:
        cached = json.loads(cpath.read_text())
    except Exception:
        return None
    if cached.get("cache_key_payload") != payload:
        return None
    if require_test and (cached.get("test_human") is None or cached.get("test_synthetic") is None):
        return None
    if require_checkpoint:
        ckpt = cached.get("checkpoint")
        if not ckpt or not Path(ckpt).is_file():
            # Coba cek di CKPT_DIR jika path berasal dari mesin lain
            fallback = CKPT_DIR / Path(ckpt).name if ckpt else None
            if fallback and fallback.is_file():
                cached["checkpoint"] = str(fallback)
            else:
                return None
    return cached


def save_json(path: Path, obj: dict | list) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2, default=str))


def store_run_cache(result: dict, payload: dict) -> Path:
    cpath = cache_file_path(payload)
    out = copy.deepcopy(result)
    out["cache_key_payload"] = payload
    out["cache_key_hash"] = cache_key_hash(payload)
    save_json(cpath, out)
    return cpath


def check_time_budget() -> None:
    if TIME_BUDGET_MIN and TIME_BUDGET_MIN > 0:
        elapsed_min = (time.time() - SESSION_START_TIME) / 60.0
        if elapsed_min >= TIME_BUDGET_MIN:
            raise TimeoutError(
                f"Batas waktu sesi tercapai ({elapsed_min:.1f} >= {TIME_BUDGET_MIN} mnt). "
                "Seluruh progres sudah tersimpan di run_cache; lanjutkan di sesi berikutnya."
            )


print(
    f"Protokol={PROTOCOL_VERSION} | max_epochs={MAX_EPOCHS} | effective_batch={EFFECTIVE_BATCH} | "
    f"RAM={system_ram_gb():.1f} GB (preload_wf={PRELOAD_WAVEFORM_IN_RAM}) | amp={AMP_DTYPE}"
)
print(
    f"MEMBER={MEMBER} -> encoder={MY_ENCODER} | owns_text_unimodal={OWNS_TEXT_UNIMODAL} | "
    f"phase3_task={SHARDS[MEMBER]['phase3_task']} | cache_files={len(list(CACHE_DIR.glob('*.json')))}"
)

In [ ]:
# =============================================================================
# CELL 5: DATASET, LEVEL FEATURES, ENCODER AUDIO/TEKS, & MODEL M3
# =============================================================================
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset


# --- 1. Dataset berbasis waveform16k + stem_mel_power ------------------------
class WaveformGainDataset(Dataset):
    """Dataset multimodal memakai cache waveform16k + stem_mel_power.

    Saat `load_waveform=False` (khusus mode `text_only`), mengembalikan dummy tensor
    kecil agar tidak membuang I/O disk/RAM, sambil tetap menyediakan `stem_mel_power`
    untuk evaluasi metrik spektral penuh di `evaluate_model`.
    """

    def __init__(
        self,
        manifest_path: Path | str,
        load_waveform: bool = True,
        preload_in_ram: bool | None = None,
    ) -> None:
        self.samples = load_manifest_records(Path(manifest_path))
        self.load_waveform = bool(load_waveform)
        if preload_in_ram is None:
            preload_in_ram = bool(PRELOAD_WAVEFORM_IN_RAM)
        self.preload_in_ram = bool(preload_in_ram and self.load_waveform)
        self._wf_cache: dict[str, np.ndarray] = {}
        if self.preload_in_ram:
            for rec in self.samples:
                ap = str(rec["audio_path"])
                if ap not in self._wf_cache:
                    self._wf_cache[ap] = np.load(waveform_path_for(ap))

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, index: int) -> dict:
        rec = self.samples[index]
        ap = str(rec["audio_path"])
        g0 = np.asarray(rec["g0_db"], dtype=np.float32)
        mask = np.asarray(rec.get("active_mask", [True] * 4), dtype=bool)
        eff_g0 = np.where(mask, g0, 0.0).astype(np.float32)

        if self.load_waveform:
            wf_raw = self._wf_cache[ap] if ap in self._wf_cache else np.load(waveform_path_for(ap))
            wf = wf_raw.astype(np.float32) * np.power(10.0, eff_g0 / 20.0)[:, None]
            wf_tensor = torch.from_numpy(wf)
        else:
            wf_tensor = torch.zeros((4, 1), dtype=torch.float32)

        stem_mel_power = RAW_STEM_MEL_POWER[ap] * np.power(10.0, eff_g0 / 10.0)[:, None]
        return {
            "id": str(rec["id"]),
            "song": str(rec["song"]),
            "split": str(rec["split"]),
            "waveform": wf_tensor,
            "stem_mel_power": torch.from_numpy(stem_mel_power.astype(np.float32)),
            "prompt": str(rec["prompt"]),
            "prompt_template": str(rec["prompt_template"]),
            "g0_db": torch.from_numpy(g0),
            "loudness_db": torch.tensor(rec["loudness_db"], dtype=torch.float32),
            "balance_target": torch.tensor(rec["balance_target"], dtype=torch.float32),
            "gains_db": torch.tensor(rec["gains_db"], dtype=torch.float32),
            "active_mask": torch.from_numpy(mask),
        }


_DATASET_CACHE: dict[tuple[str, bool], WaveformGainDataset] = {}


def get_dataset(manifest_key: str, load_waveform: bool = True) -> WaveformGainDataset:
    key = (manifest_key, bool(load_waveform))
    if key not in _DATASET_CACHE:
        _DATASET_CACHE[key] = WaveformGainDataset(CONFIG[manifest_key], load_waveform=load_waveform)
    return _DATASET_CACHE[key]


# --- 2. Fitur level eksplisit dari waveform 16 kHz (7 fitur/stem) ------------
N_LEVEL_FEATS = 7


def _assemble_level(rms_db: torch.Tensor, bands: torch.Tensor, peak_db: torch.Tensor) -> torch.Tensor:
    crest = peak_db - rms_db
    active = (rms_db > ACTIVE_THRESHOLD_DB).to(rms_db.dtype)
    stacked = torch.stack(
        [rms_db, bands[..., 0], bands[..., 1], bands[..., 2], peak_db, crest, active * 30.0], dim=-1
    )
    return stacked / 30.0


def compute_level_features(waveform: torch.Tensor, sample_rate: int = 16000) -> torch.Tensor:
    x = waveform.float()
    rms_db = 20.0 * torch.log10(x.pow(2).mean(dim=-1).clamp_min(1e-12).sqrt())
    peak_db = 20.0 * torch.log10(x.abs().amax(dim=-1).clamp_min(1e-12))
    spec = torch.fft.rfft(x, dim=-1)
    power = spec.real.pow(2) + spec.imag.pow(2)
    freqs = torch.fft.rfftfreq(x.shape[-1], d=1.0 / sample_rate).to(x.device)
    band_db = []
    for lo, hi in ((0, 250), (250, 2000), (2000, sample_rate // 2)):
        sel = (freqs >= lo) & (freqs < hi)
        band_db.append(10.0 * torch.log10(power[..., sel].mean(dim=-1).clamp_min(1e-12)))
    return _assemble_level(rms_db, torch.stack(band_db, dim=-1), peak_db)


# --- 3. Cabang Teks (IndoBERT fine-tunable) ----------------------------------
class TrainableIndoBertTextBranch(nn.Module):
    is_pretrained = True

    def __init__(self, model_name: str, hidden_dim: int = 128, dropout: float = 0.15, max_length: int = 64):
        super().__init__()
        from transformers import AutoModel, AutoTokenizer

        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name)
        self.max_length = max_length
        embed_dim = int(self.model.config.hidden_size)
        self.head = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 4),
        )

    def forward(self, prompts, device) -> torch.Tensor:
        encoded = self.tokenizer(
            list(prompts), padding=True, truncation=True, max_length=self.max_length, return_tensors="pt"
        )
        encoded = {k: v.to(device) for k, v in encoded.items()}
        hidden = self.model(**encoded).last_hidden_state
        mask = encoded["attention_mask"].unsqueeze(-1).to(hidden.dtype)
        pooled = (hidden * mask).sum(1) / mask.sum(1).clamp_min(1.0)
        return 6.0 * torch.tanh(self.head(pooled) / 6.0)


# --- 4. Registry 4 Audio Encoder (cnn_scratch, ast, mert, beats) -------------
BEATS_URL = "https://huggingface.co/lpepino/beats_ckpts/resolve/main/BEATs_iter3_plus_AS2M.pt"
BEATS_REPO_DIR = REPO_DIR / "third_party" / "unilm"
BEATS_PACKAGE_DIR = BEATS_REPO_DIR / "beats"


def download_beats_checkpoint(path: Path) -> Path:
    path = Path(path)
    if path.is_file():
        return path
    path.parent.mkdir(parents=True, exist_ok=True)
    print(f"Mengunduh checkpoint BEATs -> {path}")
    from urllib.request import urlretrieve
    urlretrieve(BEATS_URL, path)
    return path


def ensure_beats() -> Path:
    if not BEATS_PACKAGE_DIR.is_dir():
        BEATS_REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
        if BEATS_REPO_DIR.exists() and not (BEATS_REPO_DIR / ".git").exists():
            shutil.rmtree(BEATS_REPO_DIR, ignore_errors=True)
        if not (BEATS_REPO_DIR / ".git").exists():
            subprocess.run(
                ["git", "clone", "--depth", "1", "https://github.com/microsoft/unilm.git", str(BEATS_REPO_DIR)],
                check=True,
            )
    if str(BEATS_PACKAGE_DIR) not in sys.path:
        sys.path.insert(0, str(BEATS_PACKAGE_DIR))
    import torchaudio  # noqa: F401
    return BEATS_PACKAGE_DIR


def set_encoder_dropout(encoder: nn.Module | None, p: float) -> None:
    if encoder is None:
        return
    p = float(p)
    for mod in encoder.modules():
        if isinstance(mod, nn.Dropout):
            mod.p = p


class ScratchWaveformEncoder(nn.Module):
    input_key = "waveform"
    is_pretrained = False

    def __init__(self, embed_dim: int = 128, n_tokens: int = 25):
        super().__init__()
        self.embed_dim = embed_dim
        self.n_tokens = n_tokens
        self.conv = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=7, stride=4, padding=3),
            nn.BatchNorm1d(32),
            nn.GELU(),
            nn.Conv1d(32, 64, kernel_size=5, stride=4, padding=2),
            nn.BatchNorm1d(64),
            nn.GELU(),
            nn.Conv1d(64, 128, kernel_size=3, stride=4, padding=1),
            nn.BatchNorm1d(128),
            nn.GELU(),
        )
        self.proj = nn.Sequential(
            nn.Linear(128, embed_dim),
            nn.LayerNorm(embed_dim),
            nn.GELU(),
        )
        self.dropout = nn.Dropout(0.0)

    def forward(self, waveform: torch.Tensor) -> torch.Tensor:
        bsz, n_stems, n_samples = waveform.shape
        x = waveform.reshape(bsz * n_stems, 1, n_samples)
        h = F.adaptive_avg_pool1d(self.conv(x), self.n_tokens).transpose(1, 2)
        h = self.dropout(self.proj(h))
        return h.reshape(bsz, n_stems, self.n_tokens, self.embed_dim)


class _HFWaveformEncoder(nn.Module):
    input_key = "waveform"
    is_pretrained = True
    sample_rate = 16000

    def _encode_waveform(self, waveforms: torch.Tensor) -> torch.Tensor:
        raise NotImplementedError

    def forward(self, waveform: torch.Tensor) -> torch.Tensor:
        bsz, n_stems, _ = waveform.shape
        flat = waveform.reshape(bsz * n_stems, waveform.shape[-1])
        hidden = self._encode_waveform(flat)
        return hidden.reshape(bsz, n_stems, hidden.shape[-2], hidden.shape[-1])


class ASTEncoder(_HFWaveformEncoder):
    def __init__(self, model_name: str = "MIT/ast-finetuned-audioset-10-10-0.4593"):
        super().__init__()
        from transformers import ASTFeatureExtractor, ASTModel
        self.extractor = ASTFeatureExtractor.from_pretrained(model_name)
        self.model = ASTModel.from_pretrained(model_name)
        self.embed_dim = int(self.model.config.hidden_size)

    def _encode_waveform(self, waveforms: torch.Tensor) -> torch.Tensor:
        feats = self.extractor(
            list(waveforms.detach().cpu().numpy()), sampling_rate=self.sample_rate, return_tensors="pt"
        )
        return self.model(feats["input_values"].to(waveforms.device)).last_hidden_state


class MERTEncoder(_HFWaveformEncoder):
    def __init__(self, model_name: str = "m-a-p/MERT-v1-95M"):
        super().__init__()
        from transformers import AutoConfig, AutoModel, Wav2Vec2FeatureExtractor
        cfg = AutoConfig.from_pretrained(model_name, trust_remote_code=True)
        if not hasattr(cfg, "conv_pos_batch_norm"):
            cfg.conv_pos_batch_norm = False
        self.model = AutoModel.from_pretrained(model_name, config=cfg, trust_remote_code=True)
        self.extractor = Wav2Vec2FeatureExtractor.from_pretrained(model_name, trust_remote_code=True)
        self.embed_dim = int(self.model.config.hidden_size)

    def _encode_waveform(self, waveforms: torch.Tensor) -> torch.Tensor:
        feats = self.extractor(
            list(waveforms.detach().cpu().numpy()),
            sampling_rate=self.extractor.sampling_rate,
            return_tensors="pt",
        )
        return self.model(feats["input_values"].to(waveforms.device)).last_hidden_state


class BEATsEncoder(_HFWaveformEncoder):
    def __init__(self, checkpoint: str | None = None):
        super().__init__()
        try:
            from beats.BEATs import BEATs, BEATsConfig
        except Exception:
            ensure_beats()
            from BEATs import BEATs, BEATsConfig
        ckpt = download_beats_checkpoint(
            Path(checkpoint) if checkpoint else Path.home() / "beats_data" / "BEATs_iter3_plus_AS2M.pt"
        )
        payload = torch.load(ckpt, map_location="cpu", weights_only=False)
        cfg = BEATsConfig(payload["cfg"])
        self.model = BEATs(cfg)
        self.model.load_state_dict(payload["model"])
        self.embed_dim = int(cfg.encoder_embed_dim)

    def _encode_waveform(self, waveforms: torch.Tensor) -> torch.Tensor:
        out = self.model.extract_features(waveforms)
        return out[0] if isinstance(out, tuple) else out


ENCODER_REGISTRY = {
    "cnn_scratch": ScratchWaveformEncoder,
    "ast": ASTEncoder,
    "mert": MERTEncoder,
    "beats": BEATsEncoder,
}


def build_encoder(name: str) -> nn.Module:
    if name not in ENCODER_REGISTRY:
        raise ValueError(f"Encoder tidak dikenal: {name!r}. Pilihan: {sorted(ENCODER_REGISTRY)}")
    return ENCODER_REGISTRY[name]()


# --- 5. Model M3 Structured Two-Branch (mendukung inference-time modality removal) ---
class EncoderProbeModel(nn.Module):
    """M3 Structured Two-Branch (`g = L_ref + r - L`).

    - Mode `bimodal`: membangun kedua cabang (audio encoder + IndoBERT).
      Saat inferensi, `model.mode` dapat diubah sementara ke `"text_only"` (hapus audio)
      atau `"audio_only"` (hapus teks) untuk Inference-Time Modality Removal.
    - Mode `audio_only`: hanya membangun cabang audio (hemat VRAM & parameter akurat).
    - Mode `text_only`: hanya membangun cabang teks (hemat VRAM & tanpa I/O audio).
    """

    audio_input_key = "waveform"

    def __init__(self, encoder_name: str, mode: str, config: dict):
        super().__init__()
        if mode not in ("bimodal", "audio_only", "text_only"):
            raise ValueError(f"Mode tidak valid: {mode}")
        self.mode = mode
        self.train_mode = mode
        self.encoder_name = "none" if mode == "text_only" else encoder_name
        d = int(config["d_model"])

        if mode != "text_only":
            self.encoder = build_encoder(encoder_name)
            self.enc_proj = nn.Linear(self.encoder.embed_dim, d)
            self.stem_embedding = nn.Parameter(torch.zeros(1, 4, 1, d))
            self.level_mlp = nn.Sequential(nn.Linear(N_LEVEL_FEATS, d), nn.GELU(), nn.Linear(d, d))
            self.stem_norm = nn.LayerNorm(d)
            self.loudness_head = nn.Sequential(nn.Linear(d, d), nn.GELU(), nn.Linear(d, 1))
        else:
            self.encoder = None
            self.enc_proj = None
            self.stem_embedding = None
            self.level_mlp = None
            self.stem_norm = None
            self.loudness_head = None

        if mode != "audio_only":
            self.text_branch = TrainableIndoBertTextBranch(
                config["text_model"], int(config["text_hidden"]), float(config["dropout_text"])
            )
        else:
            self.text_branch = None

    def forward(self, audio_input, prompts, active_mask=None, clamp_output: bool = False) -> dict:
        device = audio_input.device
        bsz = audio_input.shape[0]
        if active_mask is not None:
            mask = active_mask.to(device=device, dtype=torch.float32)
        else:
            mask = torch.ones((bsz, 4), device=device)
        count = mask.sum(dim=1, keepdim=True).clamp_min(1.0)

        l_pred = torch.zeros((bsz, 4), device=device)
        if self.mode != "text_only" and self.encoder is not None:
            level = compute_level_features(audio_input)
            tokens = self.encoder(audio_input)
            h = self.enc_proj(tokens) + self.stem_embedding + self.level_mlp(level).unsqueeze(2)
            l_pred = self.loudness_head(self.stem_norm(h.mean(dim=2))).squeeze(-1)
        l_ref = (l_pred * mask).sum(dim=1, keepdim=True) / count

        r_pred = torch.zeros((bsz, 4), device=device)
        if self.mode != "audio_only" and self.text_branch is not None:
            raw_r = self.text_branch(prompts, device)
            r_pred = (raw_r - (raw_r * mask).sum(dim=1, keepdim=True) / count) * mask

        g_pred = (l_ref + r_pred - l_pred) * mask
        if clamp_output or not self.training:
            g_pred = g_pred.clamp(-18.0, 12.0)
        return {"gains_pred": g_pred, "loudness_pred": l_pred, "balance_pred": r_pred}

    def build_param_groups(self, config: dict) -> list[dict]:
        groups: list[dict] = []
        head_params: list[nn.Parameter] = []

        if self.encoder is not None:
            enc_lr = float(config["lr_audio"] if getattr(self.encoder, "is_pretrained", False) else config["lr_scratch"])
            groups.append({
                "params": list(self.encoder.parameters()),
                "lr": enc_lr,
                "weight_decay": float(config["wd_audio"]),
            })
            for mod in (self.enc_proj, self.level_mlp, self.stem_norm, self.loudness_head):
                head_params.extend(mod.parameters())
            head_params.append(self.stem_embedding)

        if self.text_branch is not None:
            groups.append({
                "params": list(self.text_branch.model.parameters()),
                "lr": float(config["lr_text"]),
                "weight_decay": float(config["wd_text"]),
            })
            head_params.extend(self.text_branch.head.parameters())

        if head_params:
            groups.append({
                "params": head_params,
                "lr": float(config["lr_head"]),
                "weight_decay": float(config["wd_head"]),
            })
        return groups


print("Dataset, encoders, dan EncoderProbeModel siap.")

In [ ]:
# =============================================================================
# CELL 6: TRAINING LOOP (MAX 5 EPOCH), CACHE WRAPPER, & INFERENCE REMOVAL
# =============================================================================
import gc


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def env_info() -> dict:
    info = {"env": ENV_NAME, "torch": torch.__version__, "device": str(DEVICE), "gpu": GPU_NAME}
    if torch.cuda.is_available():
        info["vram_gb"] = round(GPU_VRAM_GB, 1)
        info["cuda"] = torch.version.cuda
    return info


def strip_per_song(metrics: dict | None) -> dict | None:
    if metrics is None:
        return None
    m = dict(metrics)
    m.pop("per_song", None)
    return m


def _make_loader(dataset: Dataset, batch_size: int, shuffle: bool, seed: int | None = None) -> DataLoader:
    kwargs: dict = {
        "batch_size": int(batch_size),
        "shuffle": bool(shuffle),
        "num_workers": NUM_WORKERS,
        "pin_memory": PIN_MEMORY and DEVICE.type == "cuda",
    }
    if NUM_WORKERS > 0:
        kwargs["persistent_workers"] = True
        kwargs["prefetch_factor"] = 2
    if shuffle and seed is not None:
        gen = torch.Generator()
        gen.manual_seed(int(seed))
        kwargs["generator"] = gen
    return DataLoader(dataset, **kwargs)


def run_one_train(
    encoder_name: str,
    mode: str,
    seed: int,
    config: dict,
    eval_test: bool = False,
    save_checkpoint_path: Path | None = None,
) -> dict:
    assert int(config["epochs"]) == MAX_EPOCHS, f"Pelanggaran protokol: epochs={config['epochs']} (wajib {MAX_EPOCHS})"
    if mode != "text_only" and encoder_name == "beats":
        ensure_beats()

    seed_everything(seed)
    need_wf = (mode != "text_only")
    train_ds = get_dataset("train_manifest", load_waveform=need_wf)
    val_ds = get_dataset("val_manifest", load_waveform=need_wf)

    model = EncoderProbeModel(encoder_name, mode, config).to(DEVICE)
    set_encoder_dropout(model.encoder, config["dropout_audio"])
    groups = model.build_param_groups(config)
    optimizer = torch.optim.AdamW(groups, weight_decay=0.0)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=MAX_EPOCHS, eta_min=float(config["eta_min"])
    )

    amp = bool(config["use_amp"] and DEVICE.type == "cuda" and encoder_name not in AMP_DISABLED_ENCODERS)
    amp_dtype = AMP_DTYPE if amp else torch.float32
    use_scaler = bool(amp and amp_dtype == torch.float16)
    scaler = torch.amp.GradScaler("cuda", enabled=use_scaler)
    all_params = [p for g in groups for p in g["params"]]

    train_loader = _make_loader(train_ds, config["batch_size"], shuffle=True, seed=seed)
    val_loader = _make_loader(val_ds, config["batch_size"], shuffle=False)
    accum = max(1, int(config["grad_accum_steps"]))
    lam_l, lam_r = AUX_LAMBDA[mode]

    best = {"val_mae": float("inf"), "epoch": 0, "state": None, "metrics": {}}
    history: list[dict] = []
    t0 = time.time()
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    need_state = bool(eval_test or save_checkpoint_path is not None)

    enc_tag = "none" if mode == "text_only" else encoder_name
    print(
        f"  [TRAIN] {enc_tag}/{mode}/seed{seed} | ep={MAX_EPOCHS} | "
        f"bs={config['batch_size']}x{accum} | amp={'ON('+str(amp_dtype).split('.')[-1]+')' if amp else 'OFF'} | "
        f"params={n_params/1e6:.2f}M"
    )

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        for step, batch in enumerate(train_loader):
            audio = batch["waveform"].to(device=DEVICE, dtype=torch.float32, non_blocking=PIN_MEMORY)
            mask = batch["active_mask"].to(device=DEVICE, non_blocking=PIN_MEMORY)
            with torch.amp.autocast(device_type=DEVICE.type, dtype=amp_dtype, enabled=amp):
                out = model(audio, batch["prompt"], active_mask=mask, clamp_output=False)
                losses = compute_multimodal_loss(
                    out,
                    batch["gains_db"].to(device=DEVICE, dtype=torch.float32, non_blocking=PIN_MEMORY),
                    batch["loudness_db"].to(device=DEVICE, dtype=torch.float32, non_blocking=PIN_MEMORY),
                    batch["balance_target"].to(device=DEVICE, dtype=torch.float32, non_blocking=PIN_MEMORY),
                    active_mask=mask,
                    lambda_loudness=lam_l,
                    lambda_balance=lam_r,
                )
                loss = losses["loss"]

            if not torch.isfinite(loss):
                raise RuntimeError(f"Non-finite loss pada {enc_tag}/{mode}/ep{epoch}/step{step}")

            scaler.scale(loss / accum).backward()
            if (step + 1) % accum == 0 or (step + 1) == len(train_loader):
                if float(config["grad_clip"]) > 0:
                    scaler.unscale_(optimizer)
                    nn.utils.clip_grad_norm_(all_params, float(config["grad_clip"]))
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)

        scheduler.step()
        val_metrics = evaluate_model(model, val_loader, device=DEVICE)
        history.append({
            "epoch": epoch,
            "val_mae_db": val_metrics["mae_db"],
            "val_rmse_db": val_metrics["rmse_db"],
            "val_band_mae_db": val_metrics["band_mae_db"],
            "val_balance_mae_db": val_metrics["balance_mae_db"],
            "val_success_1db": val_metrics["success_rate_1db"],
            "val_adherence": val_metrics["instruction_adherence"],
        })
        print(
            f"    ep {epoch:02d}/{MAX_EPOCHS} | val_MAE={val_metrics['mae_db']:.4f} dB | "
            f"RMSE={val_metrics['rmse_db']:.4f} | band={val_metrics['band_mae_db']:.4f} | "
            f"adh={val_metrics['instruction_adherence']:.1f}%"
        )

        if val_metrics["mae_db"] < best["val_mae"] - 1e-4:
            state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()} if need_state else None
            best = {"val_mae": val_metrics["mae_db"], "epoch": epoch, "state": state, "metrics": val_metrics}
        elif int(config["patience"]) > 0 and (epoch - best["epoch"]) >= int(config["patience"]):
            print(f"    early stop pada epoch {epoch} (best epoch {best['epoch']})")
            break

    if need_state and best["state"] is not None:
        model.load_state_dict(best["state"])

    test_human = None
    test_synthetic = None
    if eval_test:
        th_loader = _make_loader(get_dataset("test_human_manifest", load_waveform=need_wf), config["batch_size"], False)
        ts_loader = _make_loader(get_dataset("test_synthetic_manifest", load_waveform=need_wf), config["batch_size"], False)
        test_human = strip_per_song(evaluate_model(model, th_loader, device=DEVICE))
        test_synthetic = strip_per_song(evaluate_model(model, ts_loader, device=DEVICE))
        print(
            f"    [TEST] human_MAE={test_human['mae_db']:.4f} dB | "
            f"synth_MAE={test_synthetic['mae_db']:.4f} dB"
        )

    ckpt_str = None
    if save_checkpoint_path is not None and best["state"] is not None:
        save_checkpoint_path = Path(save_checkpoint_path)
        save_checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
        torch.save({
            "state_dict": best["state"],
            "config": config,
            "encoder": enc_tag,
            "mode": mode,
            "seed": seed,
            "protocol_version": PROTOCOL_VERSION,
            "best_epoch": best["epoch"],
            "validation": strip_per_song(best["metrics"]),
            "test_human": test_human,
            "test_synthetic": test_synthetic,
        }, save_checkpoint_path)
        ckpt_str = str(save_checkpoint_path)

    train_sec = round(time.time() - t0, 1)
    del model, optimizer, scheduler, scaler
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        "protocol_version": PROTOCOL_VERSION,
        "encoder": enc_tag,
        "mode": mode,
        "seed": int(seed),
        "member": MEMBER,
        "env": env_info(),
        "trainable_params": int(n_params),
        "train_sec": train_sec,
        "epochs_run": len(history),
        "best_epoch": int(best["epoch"]),
        "validation": strip_per_song(best["metrics"]),
        "test_human": test_human,
        "test_synthetic": test_synthetic,
        "checkpoint": ckpt_str,
        "history": history,
    }


def run_cached(
    encoder_name: str,
    mode: str,
    seed: int,
    overrides: dict | None = None,
    eval_test: bool = False,
    save_checkpoint_path: Path | None = None,
    tag: str = "",
) -> tuple[dict, bool]:
    """Jalankan training dengan Strict Model Caching + OOM retry (effective_batch=16).

    Returns: `(result_dict, cache_hit_bool)`.
    """
    cfg = make_config(overrides=overrides, encoder_name=encoder_name, mode=mode)
    payload = canonical_cache_payload(encoder_name, mode, seed, cfg)
    cached = lookup_run_cache(
        encoder_name, mode, seed, cfg,
        require_test=eval_test,
        require_checkpoint=(save_checkpoint_path is not None),
    )
    if cached is not None:
        print(
            f"[CACHE HIT] {tag or payload['encoder']+'/'+mode} | key={cache_key_hash(payload)} | "
            f"val_MAE={cached['validation']['mae_db']:.4f} dB"
        )
        return cached, True

    check_time_budget()
    print(f"[CACHE MISS] {tag or payload['encoder']+'/'+mode} | key={cache_key_hash(payload)}")

    bs_start = int(cfg["batch_size"])
    candidates = [b for b in (bs_start, 8, 4, 2) if b <= bs_start and EFFECTIVE_BATCH % b == 0]
    candidates = list(dict.fromkeys(candidates))
    last_exc = None
    for idx, cand_bs in enumerate(candidates):
        try_cfg = copy.deepcopy(cfg)
        try_cfg["batch_size"] = cand_bs
        try_cfg["grad_accum_steps"] = EFFECTIVE_BATCH // cand_bs
        if idx > 0:
            print(f"  [OOM retry] batch_size={cand_bs} x grad_accum={try_cfg['grad_accum_steps']} (effective={EFFECTIVE_BATCH})")
        try:
            result = run_one_train(
                encoder_name, mode, seed, try_cfg,
                eval_test=eval_test,
                save_checkpoint_path=save_checkpoint_path,
            )
            store_run_cache(result, payload)
            sync_to_drive(include_checkpoints=(save_checkpoint_path is not None))
            return result, False
        except torch.cuda.OutOfMemoryError as exc:
            last_exc = exc
            gc.collect()
            torch.cuda.empty_cache()
    raise last_exc


@torch.no_grad()
def evaluate_checkpoint_modality_removal(checkpoint_path: Path | str, eval_mode: str) -> dict:
    """Inference-Time Modality Removal pada checkpoint Bimodal Final (0 training run).

    - `eval_mode="text_only"`  -> Audio modality dihapus (`L_pred = 0`, `g = r`).
    - `eval_mode="audio_only"` -> Text modality dihapus (`r_pred = 0`, `g = L_ref - L`).
    """
    assert eval_mode in ("text_only", "audio_only"), f"eval_mode harus text_only/audio_only, bukan {eval_mode}"
    ckpt = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    enc = ckpt["encoder"]
    cfg = ckpt["config"]
    if enc == "beats":
        ensure_beats()

    model = EncoderProbeModel(enc, "bimodal", cfg).to(DEVICE)
    model.load_state_dict(ckpt["state_dict"])
    model.eval()
    model.mode = eval_mode  # Matikan 1 cabang tanpa retrain!

    need_wf = (eval_mode != "text_only")
    bs = int(cfg.get("batch_size", 4))
    th_loader = _make_loader(get_dataset("test_human_manifest", load_waveform=need_wf), bs, False)
    ts_loader = _make_loader(get_dataset("test_synthetic_manifest", load_waveform=need_wf), bs, False)

    t0 = time.time()
    test_human = strip_per_song(evaluate_model(model, th_loader, device=DEVICE))
    test_synthetic = strip_per_song(evaluate_model(model, ts_loader, device=DEVICE))
    elapsed = round(time.time() - t0, 1)

    print(
        f"  [INFERENCE REMOVAL -> {eval_mode}] ckpt={Path(checkpoint_path).name} ({elapsed}s) | "
        f"human_MAE={test_human['mae_db']:.4f} dB | synth_MAE={test_synthetic['mae_db']:.4f} dB"
    )

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        "protocol_version": PROTOCOL_VERSION,
        "encoder": enc,
        "source_checkpoint": str(checkpoint_path),
        "seed": int(ckpt["seed"]),
        "inference_mode": eval_mode,
        "eval_sec": elapsed,
        "test_human": test_human,
        "test_synthetic": test_synthetic,
    }


print("Engine training, strict model caching, dan inference-time modality removal siap.")

In [ ]:
# =============================================================================
# CELL 7: ORKESTRATOR EKSPERIMEN (BASELINE + 4 STAGE OFAT + ABLATION AKHIR TEST)
# =============================================================================
import csv


def _base_audio_params(encoder_name: str) -> dict:
    lr_key = "lr_scratch" if encoder_name == "cnn_scratch" else "lr_audio"
    return {
        lr_key: float(CONFIG[lr_key]),
        "wd_audio": float(CONFIG["wd_audio"]),
    }


def _base_text_params() -> dict:
    return {
        "lr_text": float(CONFIG["lr_text"]),
        "text_hidden": int(CONFIG["text_hidden"]),
    }


def _write_csv(path: Path, rows: list[dict]) -> None:
    if not rows:
        return
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)


def _mean_std(vals: list[float]) -> tuple[float, float]:
    arr = np.asarray(vals, dtype=float)
    return float(arr.mean()), float(arr.std(ddof=0))


def _summarize_seed_runs(
    name: str,
    encoder: str,
    mode: str,
    seed_runs: list[dict],
    params: dict,
    extra: dict | None = None,
) -> dict:
    summary = {
        "protocol_version": PROTOCOL_VERSION,
        "name": name,
        "encoder": encoder,
        "mode": mode,
        "params": params,
        "n_seeds": len(seed_runs),
        "seeds": [int(r["seed"]) for r in seed_runs],
    }
    if extra:
        summary.update(extra)

    metric_keys = [
        "mae_db", "rmse_db", "r2", "band_mae_db", "band_mse_norm",
        "log_spectral_distance_db", "success_rate_1db",
        "balance_mae_db", "intent_direction_acc", "instruction_adherence",
    ]
    for split_key in ("validation", "test_human", "test_synthetic"):
        split_items = [r[split_key] for r in seed_runs if r.get(split_key)]
        if not split_items:
            continue
        agg: dict[str, object] = {}
        for mk in metric_keys:
            vals = [float(item[mk]) for item in split_items if mk in item]
            if vals:
                mu, sd = _mean_std(vals)
                agg[f"{mk}_mean"] = round(mu, 4)
                agg[f"{mk}_std"] = round(sd, 4)
        # Per-stem MAE mean +- std
        stem_agg = {}
        for stem in STEM_NAMES:
            s_vals = [float(item["mae_per_stem_db"][stem]) for item in split_items if "mae_per_stem_db" in item]
            if s_vals:
                mu, sd = _mean_std(s_vals)
                stem_agg[stem] = {"mean": round(mu, 4), "std": round(sd, 4)}
        if stem_agg:
            agg["mae_per_stem_db"] = stem_agg
        summary[split_key] = agg

    summary["per_seed"] = [
        {
            "seed": int(r["seed"]),
            "best_epoch": r.get("best_epoch"),
            "train_sec": r.get("train_sec", r.get("eval_sec")),
            "checkpoint": r.get("checkpoint", r.get("source_checkpoint")),
            "val_mae_db": r.get("validation", {}).get("mae_db") if r.get("validation") else None,
            "test_human_mae_db": r.get("test_human", {}).get("mae_db") if r.get("test_human") else None,
            "test_synthetic_mae_db": r.get("test_synthetic", {}).get("mae_db") if r.get("test_synthetic") else None,
        }
        for r in seed_runs
    ]
    return summary


# -----------------------------------------------------------------------------
# PHASE 1A: Ablation Awal & Baseline (Default CONFIG, Val, Seed 42)
# -----------------------------------------------------------------------------
def run_baseline_phase1() -> None:
    print(f"\n================ ABLATION AWAL & BASELINE (MEMBER {MEMBER}: {MY_ENCODER}) ================")
    seed = int(CONFIG["tuning_seed"])

    if OWNS_TEXT_UNIMODAL:
        base_text = _base_text_params()
        res, hit = run_cached("none", "text_only", seed, overrides=base_text, tag="baseline/text_only")
        save_json(BASELINE_DIR / "baseline_text_only.json", {
            "protocol_version": PROTOCOL_VERSION,
            "encoder": "none",
            "mode": "text_only",
            "member": MEMBER,
            "seed": seed,
            "baseline_params": base_text,
            "val_mae_db": res["validation"]["mae_db"],
            "validation": res["validation"],
            "best_epoch": res["best_epoch"],
            "train_sec": res["train_sec"],
            "cache_hit": hit,
            "cache_key_hash": res.get("cache_key_hash"),
        })

    base_audio = _base_audio_params(MY_ENCODER)
    base_bimodal = {**base_audio, **_base_text_params()}
    for mode, params in (("audio_only", base_audio), ("bimodal", base_bimodal)):
        res, hit = run_cached(MY_ENCODER, mode, seed, overrides=params, tag=f"baseline/{MY_ENCODER}/{mode}")
        save_json(BASELINE_DIR / f"baseline_{mode}_{MY_ENCODER}.json", {
            "protocol_version": PROTOCOL_VERSION,
            "encoder": MY_ENCODER,
            "mode": mode,
            "member": MEMBER,
            "seed": seed,
            "baseline_params": params,
            "val_mae_db": res["validation"]["mae_db"],
            "validation": res["validation"],
            "best_epoch": res["best_epoch"],
            "train_sec": res["train_sec"],
            "cache_hit": hit,
            "cache_key_hash": res.get("cache_key_hash"),
        })


# -----------------------------------------------------------------------------
# Helper Generik OFAT (dengan Strict Model Caching sehingga baseline tidak diulang)
# -----------------------------------------------------------------------------
def run_ofat_sequence(
    stage_label: str,
    encoder_name: str,
    mode: str,
    fixed_params: dict,
    init_tuned_params: dict,
    knob_sequence: list[tuple[str, list]],
) -> dict:
    seed = int(CONFIG["tuning_seed"])
    best_tuned = dict(init_tuned_params)
    best_res = None
    trials: list[dict] = []

    # Catat skor baseline terlebih dahulu (selalu CACHE HIT setelah Ablation Awal / Stage 3)
    base_full = {**fixed_params, **init_tuned_params}
    base_res, _ = run_cached(
        encoder_name, mode, seed, overrides=base_full,
        tag=f"{stage_label}/baseline_ref",
    )

    for step_idx, (knob, values) in enumerate(knob_sequence, start=1):
        stage_anchor = dict(best_tuned)
        step_best_val = stage_anchor[knob]
        step_best_mae = float("inf")
        for val in values:
            cand_tuned = dict(stage_anchor)
            cand_tuned[knob] = val
            full_params = {**fixed_params, **cand_tuned}
            res, hit = run_cached(
                encoder_name, mode, seed, overrides=full_params,
                tag=f"{stage_label}/s{step_idx}_{knob}={val}",
            )
            val_mae = float(res["validation"]["mae_db"])
            trials.append({
                "stage": stage_label,
                "encoder": "none" if mode == "text_only" else encoder_name,
                "mode": mode,
                "ofat_step": step_idx,
                "knob": knob,
                "value": val,
                "val_mae_db": val_mae,
                "val_rmse_db": float(res["validation"]["rmse_db"]),
                "val_band_mae_db": float(res["validation"]["band_mae_db"]),
                "val_balance_mae_db": float(res["validation"]["balance_mae_db"]),
                "best_epoch": int(res["best_epoch"]),
                "train_sec": float(res["train_sec"]),
                "cache_hit": bool(hit),
                "params_json": json.dumps(full_params, sort_keys=True),
            })
            if val_mae < step_best_mae:
                step_best_mae = val_mae
                step_best_val = val
        best_tuned = dict(stage_anchor)
        best_tuned[knob] = step_best_val

    best_full = {**fixed_params, **best_tuned}
    # Ambil hasil terbaik dari cache (100% CACHE HIT)
    best_res, _ = run_cached(encoder_name, mode, seed, overrides=best_full, tag=f"{stage_label}/confirm_best")
    return {
        "protocol_version": PROTOCOL_VERSION,
        "stage": stage_label,
        "encoder": "none" if mode == "text_only" else encoder_name,
        "mode": mode,
        "member": MEMBER,
        "seed": seed,
        "baseline_params": base_full,
        "baseline_val_mae_db": float(base_res["validation"]["mae_db"]),
        "best_tuned_params": best_tuned,
        "best_full_params": best_full,
        "best_val_mae_db": float(best_res["validation"]["mae_db"]),
        "best_validation": best_res["validation"],
        "delta_mae_db": round(float(best_res["validation"]["mae_db"]) - float(base_res["validation"]["mae_db"]), 4),
        "n_trials_logged": len(trials),
        "n_cache_hits": sum(1 for t in trials if t["cache_hit"]),
        "trials": trials,
    }


# -----------------------------------------------------------------------------
# STAGE 1: Audio Unimodal OFAT (tiap audio encoder)
# -----------------------------------------------------------------------------
def run_stage1_audio_unimodal() -> dict:
    print(f"\n================ STAGE 1: AUDIO UNIMODAL OFAT ({MY_ENCODER}) ================")
    space = AUDIO_SEARCH_SPACE["scratch" if MY_ENCODER == "cnn_scratch" else "pretrained"]
    lr_key = "lr_scratch" if MY_ENCODER == "cnn_scratch" else "lr_audio"
    summary = run_ofat_sequence(
        stage_label="stage1_audio_unimodal",
        encoder_name=MY_ENCODER,
        mode="audio_only",
        fixed_params={},
        init_tuned_params=_base_audio_params(MY_ENCODER),
        knob_sequence=[(lr_key, space[lr_key]), ("wd_audio", space["wd_audio"])],
    )
    save_json(STAGES_DIR / f"stage1_audio_unimodal_{MY_ENCODER}.json", summary)
    _write_csv(STAGES_DIR / f"stage1_ofat_{MY_ENCODER}.csv", summary["trials"])
    print(
        f"[STAGE 1 DONE] {MY_ENCODER} audio_only | base={summary['baseline_val_mae_db']:.4f} -> "
        f"best={summary['best_val_mae_db']:.4f} dB | params={summary['best_tuned_params']}"
    )
    return summary


# -----------------------------------------------------------------------------
# STAGE 2: Text Unimodal OFAT (sekali, dimiliki MEMBER 1)
# -----------------------------------------------------------------------------
def run_stage2_text_unimodal() -> dict | None:
    if not OWNS_TEXT_UNIMODAL:
        print(f"\n[STAGE 2] Dilewati: text_only dikerjakan oleh MEMBER 1 (member aktif: {MEMBER}).")
        return None
    print("\n================ STAGE 2: TEXT UNIMODAL OFAT (IndoBERT) ================")
    summary = run_ofat_sequence(
        stage_label="stage2_text_unimodal",
        encoder_name="none",
        mode="text_only",
        fixed_params={},
        init_tuned_params=_base_text_params(),
        knob_sequence=[("lr_text", TEXT_SEARCH_SPACE["lr_text"]), ("text_hidden", TEXT_SEARCH_SPACE["text_hidden"])],
    )
    save_json(STAGES_DIR / "stage2_text_unimodal.json", summary)
    _write_csv(STAGES_DIR / "stage2_ofat_text.csv", summary["trials"])
    print(
        f"[STAGE 2 DONE] text_only | base={summary['baseline_val_mae_db']:.4f} -> "
        f"best={summary['best_val_mae_db']:.4f} dB | params={summary['best_tuned_params']}"
    )
    return summary


# -----------------------------------------------------------------------------
# STAGE 3: Bimodal Audio OFAT (Text ditrain dgn baseline params, Audio di-OFAT)
# -----------------------------------------------------------------------------
def run_stage3_bimodal_audio() -> dict:
    print(f"\n================ STAGE 3: BIMODAL AUDIO OFAT ({MY_ENCODER}) ================")
    space = AUDIO_SEARCH_SPACE["scratch" if MY_ENCODER == "cnn_scratch" else "pretrained"]
    lr_key = "lr_scratch" if MY_ENCODER == "cnn_scratch" else "lr_audio"
    summary = run_ofat_sequence(
        stage_label="stage3_bimodal_audio",
        encoder_name=MY_ENCODER,
        mode="bimodal",
        fixed_params=_base_text_params(),
        init_tuned_params=_base_audio_params(MY_ENCODER),
        knob_sequence=[(lr_key, space[lr_key]), ("wd_audio", space["wd_audio"])],
    )
    save_json(STAGES_DIR / f"stage3_bimodal_audio_{MY_ENCODER}.json", summary)
    _write_csv(STAGES_DIR / f"stage3_ofat_{MY_ENCODER}.csv", summary["trials"])
    print(
        f"[STAGE 3 DONE] {MY_ENCODER} bimodal | base={summary['baseline_val_mae_db']:.4f} -> "
        f"best={summary['best_val_mae_db']:.4f} dB | audio_params={summary['best_tuned_params']}"
    )
    return summary


# -----------------------------------------------------------------------------
# Seleksi Pemenang Stage 1, Top-2 Stage 3, & Final Winner Stage 4
# -----------------------------------------------------------------------------
def load_stage_files(prefix: str) -> dict[str, dict]:
    out = {}
    for enc in CONFIG["encoders"]:
        p = STAGES_DIR / f"{prefix}_{enc}.json"
        if p.is_file():
            d = json.loads(p.read_text())
            if d.get("protocol_version") == PROTOCOL_VERSION:
                out[enc] = d
    return out


def get_stage1_winner() -> dict | None:
    s1 = load_stage_files("stage1_audio_unimodal")
    if len(s1) != len(CONFIG["encoders"]):
        return None
    return min(s1.values(), key=lambda d: float(d["best_val_mae_db"]))


def get_stage3_top2() -> list[dict] | None:
    s3 = load_stage_files("stage3_bimodal_audio")
    if len(s3) != len(CONFIG["encoders"]):
        return None
    ranked = sorted(s3.values(), key=lambda d: float(d["best_val_mae_db"]))
    return ranked[:2]


def get_stage4_winner() -> dict | None:
    top2 = get_stage3_top2()
    if top2 is None:
        return None
    s4_candidates = []
    for item in top2:
        p = STAGES_DIR / f"stage4_bimodal_text_{item['encoder']}.json"
        if not p.is_file():
            return None
        d = json.loads(p.read_text())
        if d.get("protocol_version") != PROTOCOL_VERSION:
            return None
        s4_candidates.append(d)
    winner = min(s4_candidates, key=lambda d: float(d["best_val_mae_db"]))
    save_json(STAGES_DIR / "stage4_winner.json", {
        "protocol_version": PROTOCOL_VERSION,
        "top2_stage3": [x["encoder"] for x in top2],
        "winner_encoder": winner["encoder"],
        "winner_val_mae_db": winner["best_val_mae_db"],
        "best_audio_params_stage3": {k: v for k, v in winner["best_full_params"].items() if k not in TEXT_SEARCH_SPACE},
        "best_text_params_stage4": winner["best_tuned_params"],
        "best_full_params": winner["best_full_params"],
        "candidates": [
            {"encoder": c["encoder"], "stage3_val_mae_db": c["baseline_val_mae_db"], "stage4_val_mae_db": c["best_val_mae_db"]}
            for c in s4_candidates
        ],
    })
    return winner


# -----------------------------------------------------------------------------
# PHASE 2 -> STAGE 4: Bimodal Text OFAT pada Top-2 Stage 3
# -----------------------------------------------------------------------------
def run_stage4_bimodal_text() -> list[dict]:
    top2 = get_stage3_top2()
    if top2 is None:
        have = sorted(load_stage_files("stage3_bimodal_audio"))
        print(f"\n[STAGE 4] Belum bisa jalan: baru ada {len(have)}/4 hasil Stage 3 ({have}). Gabungkan hasil Phase 1 dulu.")
        return []

    top2_names = [x["encoder"] for x in top2]
    print(f"\n================ STAGE 4: BIMODAL TEXT OFAT | TOP-2 STAGE 3 = {top2_names} ================")
    if STAGE4_ENCODERS is None:
        target_encoders = [MY_ENCODER] if MY_ENCODER in top2_names else []
    elif STAGE4_ENCODERS == "all_top2" or "all_top2" in STAGE4_ENCODERS:
        target_encoders = list(top2_names)
    else:
        target_encoders = [e for e in STAGE4_ENCODERS if e in top2_names]

    if not target_encoders:
        print(f"[STAGE 4] Encoder MEMBER {MEMBER} ({MY_ENCODER}) tidak masuk Top-2 ({top2_names}). Tidak perlu training.")
        get_stage4_winner()
        return []

    summaries = []
    for enc in target_encoders:
        s3_entry = next(x for x in top2 if x["encoder"] == enc)
        fixed_audio_s3 = dict(s3_entry["best_tuned_params"])
        summary = run_ofat_sequence(
            stage_label="stage4_bimodal_text",
            encoder_name=enc,
            mode="bimodal",
            fixed_params=fixed_audio_s3,
            init_tuned_params=_base_text_params(),
            knob_sequence=[("lr_text", TEXT_SEARCH_SPACE["lr_text"]), ("text_hidden", TEXT_SEARCH_SPACE["text_hidden"])],
        )
        save_json(STAGES_DIR / f"stage4_bimodal_text_{enc}.json", summary)
        _write_csv(STAGES_DIR / f"stage4_ofat_text_{enc}.csv", summary["trials"])
        print(
            f"[STAGE 4 DONE] {enc} bimodal | s3_mae={summary['baseline_val_mae_db']:.4f} -> "
            f"s4_mae={summary['best_val_mae_db']:.4f} dB | text_params={summary['best_tuned_params']}"
        )
        summaries.append(summary)

    winner = get_stage4_winner()
    if winner is not None:
        print(f"[STAGE 4 WINNER] {winner['encoder']} (val_MAE={winner['best_val_mae_db']:.4f} dB)")
    return summaries


# -----------------------------------------------------------------------------
# PHASE 3 -> ABLATION AKHIR TEST (3 Seed + 2 Inference-Time Modality Removal)
# -----------------------------------------------------------------------------
def run_phase3_test_ablation() -> None:
    s1_winner = get_stage1_winner()
    s2_path = STAGES_DIR / "stage2_text_unimodal.json"
    s4_winner = get_stage4_winner()

    missing_prereqs = []
    if s1_winner is None:
        missing_prereqs.append("Stage 1 (4/4 encoder)")
    if not s2_path.is_file():
        missing_prereqs.append("Stage 2 (text_only)")
    if s4_winner is None:
        missing_prereqs.append("Stage 4 (2/2 Top-2 encoder)")
    if missing_prereqs:
        print(f"\n[PHASE 3] Belum bisa jalan, prasyarat kurang: {', '.join(missing_prereqs)}")
        return

    s2_best = json.loads(s2_path.read_text())
    seeds = [int(s) for s in CONFIG["final_seeds"]]
    win_enc_s4 = s4_winner["encoder"]
    win_enc_s1 = s1_winner["encoder"]

    if PHASE3_TASKS is None:
        tasks = [SHARDS[MEMBER]["phase3_task"]]
    elif PHASE3_TASKS == "all" or "all" in PHASE3_TASKS:
        tasks = ["unimodal_audio", "unimodal_text", "bimodal_baseline", "bimodal_final"]
    else:
        tasks = list(PHASE3_TASKS)

    print(
        f"\n================ PHASE 3: ABLATION AKHIR TEST (3 SEEDS={seeds}) ================\n"
        f"  Pemenang Stage 1 (Audio Unimodal): {win_enc_s1}\n"
        f"  Pemenang Stage 4 (Bimodal Final) : {win_enc_s4}\n"
        f"  Tugas MEMBER {MEMBER}            : {tasks}"
    )

    # 1. Unimodal Audio (best Stage 1, 3 seed)
    if "unimodal_audio" in tasks:
        print(f"\n--- [Test 1/6] Unimodal Audio (Stage 1 Winner: {win_enc_s1}) ---")
        runs = []
        for s in seeds:
            r, _ = run_cached(
                win_enc_s1, "audio_only", s,
                overrides=s1_winner["best_full_params"],
                eval_test=True,
                tag=f"test/unimodal_audio/{win_enc_s1}/seed{s}",
            )
            runs.append(r)
        summary = _summarize_seed_runs(
            "Unimodal Audio (Stage 1 Best)", win_enc_s1, "audio_only", runs,
            s1_winner["best_full_params"], extra={"source_stage": "stage1"},
        )
        save_json(TEST_DIR / "test_ablation_unimodal_audio.json", summary)

        # Opsional: jika encoder pemenang Stage 1 != Stage 4, tambahkan controlled audio_only untuk encoder Stage 4
        if ALSO_TEST_STAGE4_ENC_AUDIO_ONLY and win_enc_s1 != win_enc_s4:
            s1_all = load_stage_files("stage1_audio_unimodal")
            s1_for_s4 = s1_all[win_enc_s4]
            runs_ctrl = []
            for s in seeds:
                r, _ = run_cached(
                    win_enc_s4, "audio_only", s,
                    overrides=s1_for_s4["best_full_params"],
                    eval_test=True,
                    tag=f"test/unimodal_audio_ctrl/{win_enc_s4}/seed{s}",
                )
                runs_ctrl.append(r)
            save_json(
                TEST_DIR / f"test_ablation_unimodal_audio_same_encoder_{win_enc_s4}.json",
                _summarize_seed_runs(
                    f"Unimodal Audio ({win_enc_s4}, Stage 1)", win_enc_s4, "audio_only", runs_ctrl,
                    s1_for_s4["best_full_params"], extra={"source_stage": "stage1_controlled"},
                ),
            )

    # 2. Unimodal Text (best Stage 2, 3 seed)
    if "unimodal_text" in tasks:
        print("\n--- [Test 2/6] Unimodal Text (Stage 2 Best: IndoBERT) ---")
        runs = []
        for s in seeds:
            r, _ = run_cached(
                "none", "text_only", s,
                overrides=s2_best["best_full_params"],
                eval_test=True,
                tag=f"test/unimodal_text/seed{s}",
            )
            runs.append(r)
        summary = _summarize_seed_runs(
            "Unimodal Text (Stage 2 Best)", "none", "text_only", runs,
            s2_best["best_full_params"], extra={"source_stage": "stage2"},
        )
        save_json(TEST_DIR / "test_ablation_unimodal_text.json", summary)

    # 3. Bimodal Baseline (Stage 4 winning encoder + default baseline params, 3 seed)
    if "bimodal_baseline" in tasks:
        print(f"\n--- [Test 3/6] Bimodal Baseline ({win_enc_s4}, Default Params) ---")
        base_params = {**_base_audio_params(win_enc_s4), **_base_text_params()}
        runs = []
        for s in seeds:
            r, _ = run_cached(
                win_enc_s4, "bimodal", s,
                overrides=base_params,
                eval_test=True,
                tag=f"test/bimodal_baseline/{win_enc_s4}/seed{s}",
            )
            runs.append(r)
        summary = _summarize_seed_runs(
            "Bimodal Baseline (Default Params)", win_enc_s4, "bimodal", runs,
            base_params, extra={"source_stage": "baseline"},
        )
        save_json(TEST_DIR / "test_ablation_bimodal_baseline.json", summary)

    # 4, 5, 6. Bimodal Final (Audio Stage 3 + Text Stage 4) + 2 Inference-Time Modality Removal
    if "bimodal_final" in tasks:
        print(f"\n--- [Test 4/6] Bimodal Final ({win_enc_s4}, Audio Stage 3 + Text Stage 4) ---")
        final_params = dict(s4_winner["best_full_params"])
        runs_final = []
        ckpts: list[Path] = []
        for s in seeds:
            # Checkpoint selalu disimpan untuk Bimodal Final karena diperlukan oleh Inference-Time Removal
            ckpt_p = CKPT_DIR / f"bimodal_final_{win_enc_s4}_seed{s}.pt"
            r, _ = run_cached(
                win_enc_s4, "bimodal", s,
                overrides=final_params,
                eval_test=True,
                save_checkpoint_path=ckpt_p,
                tag=f"test/bimodal_final/{win_enc_s4}/seed{s}",
            )
            runs_final.append(r)
            ckpts.append(Path(r["checkpoint"]))

        save_json(
            TEST_DIR / "test_ablation_bimodal_final.json",
            _summarize_seed_runs(
                "Bimodal Final (Stage 3 + Stage 4)", win_enc_s4, "bimodal", runs_final,
                final_params, extra={"source_stage": "stage3+stage4"},
            ),
        )

        # 5 & 6. Inference-time Modality Removal (idempotent, 0 training run)
        for idx_lbl, out_name, variant_title, eval_mode, rem_mod in (
            ("5/6", "test_ablation_bimodal_text_only_inference.json", "Bimodal Text Only (Inference Audio Removal)", "text_only", "audio"),
            ("6/6", "test_ablation_bimodal_audio_only_inference.json", "Bimodal Audio Only (Inference Text Removal)", "audio_only", "text"),
        ):
            out_p = TEST_DIR / out_name
            skip_inf = False
            if out_p.is_file() and not FORCE_RERUN:
                try:
                    prev = json.loads(out_p.read_text())
                    if (
                        prev.get("protocol_version") == PROTOCOL_VERSION
                        and prev.get("encoder") == win_enc_s4
                        and prev.get("seeds") == seeds
                        and prev.get("params") == final_params
                    ):
                        skip_inf = True
                except Exception:
                    skip_inf = False
            if skip_inf:
                print(f"\n--- [Test {idx_lbl}] {variant_title} [CACHE HIT: {out_name}] ---")
                continue
            print(f"\n--- [Test {idx_lbl}] {variant_title} (0 train) ---")
            rem_runs = [evaluate_checkpoint_modality_removal(cp, eval_mode) for cp in ckpts]
            save_json(
                out_p,
                _summarize_seed_runs(
                    variant_title, win_enc_s4, f"bimodal->{eval_mode}",
                    rem_runs, final_params,
                    extra={"source_stage": "stage4_inference_removal", "removed_modality": rem_mod},
                ),
            )

    sync_to_drive(include_checkpoints=EXPORT_FINAL_CHECKPOINTS)
    print("\n[PHASE 3] Tugas untuk MEMBER ini selesai.")


# -----------------------------------------------------------------------------
# EKSEKUSI UTAMA BERDASARKAN `PHASE`
# -----------------------------------------------------------------------------
try:
    if PHASE == "phase1":
        run_baseline_phase1()
        run_stage1_audio_unimodal()
        run_stage2_text_unimodal()
        run_stage3_bimodal_audio()
    elif PHASE == "phase2":
        run_stage4_bimodal_text()
    elif PHASE == "phase3":
        run_phase3_test_ablation()
    elif PHASE == "auto":
        run_baseline_phase1()
        run_stage1_audio_unimodal()
        run_stage2_text_unimodal()
        run_stage3_bimodal_audio()
        run_stage4_bimodal_text()
        run_phase3_test_ablation()
    elif PHASE == "report_only":
        print("Mode report_only: melewati seluruh training, langsung membuat tabel & grafik.")
except TimeoutError as exc:
    print(f"\n[TIME GUARD] {exc}")
finally:
    sync_to_drive(include_checkpoints=EXPORT_FINAL_CHECKPOINTS)

In [ ]:
# =============================================================================
# CELL 8: AGREGASI HASIL & TABEL CSV (BASELINE, STAGE 1-4, & ABLATION AKHIR TEST)
# =============================================================================
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 25)
pd.set_option("display.width", 140)

# --- 1. Tabel Baseline Awal (Validation) -------------------------------------
baseline_rows = []
for p in sorted(BASELINE_DIR.glob("baseline_*.json")):
    d = json.loads(p.read_text())
    if d.get("protocol_version") != PROTOCOL_VERSION:
        continue
    v = d["validation"]
    baseline_rows.append({
        "mode": d["mode"],
        "encoder": d["encoder"],
        "val_mae_db": v["mae_db"],
        "val_rmse_db": v["rmse_db"],
        "val_band_mae_db": v["band_mae_db"],
        "val_balance_mae_db": v["balance_mae_db"],
        "val_adherence_pct": v["instruction_adherence"],
        "best_epoch": d["best_epoch"],
        "train_sec": d["train_sec"],
        "baseline_params": json.dumps(d["baseline_params"], sort_keys=True),
    })

df_baseline = pd.DataFrame(baseline_rows)
if not df_baseline.empty:
    df_baseline = df_baseline.sort_values(["mode", "val_mae_db"]).reset_index(drop=True)
    df_baseline.to_csv(REPORT_DIR / "table1_baseline_ablation.csv", index=False)
    print("=== TABEL 1: ABLATION AWAL & BASELINE (VALIDATION, SEED 42) ===")
    display(df_baseline)
else:
    print("Tabel 1: belum ada hasil baseline.")

# --- 2. Tabel Semua Trial OFAT (Stage 1 - Stage 4) ---------------------------
all_trials = []
for p in sorted(STAGES_DIR.glob("stage*_*.json")):
    if p.name == "stage4_winner.json":
        continue
    d = json.loads(p.read_text())
    if d.get("protocol_version") != PROTOCOL_VERSION:
        continue
    all_trials.extend(d.get("trials", []))

df_ofat = pd.DataFrame(all_trials)
if not df_ofat.empty:
    df_ofat.to_csv(REPORT_DIR / "table2_ofat_all_stages.csv", index=False)
    print(f"\n=== TABEL 2: LOG OFAT STAGE 1-4 ({len(df_ofat)} evaluasi, {int(df_ofat['cache_hit'].sum())} cache hit) ===")
    display(df_ofat[["stage", "encoder", "mode", "knob", "value", "val_mae_db", "val_band_mae_db", "best_epoch", "cache_hit"]])

# --- 3. Tabel Ringkasan Baseline vs Tuned per Stage (Validation) -------------
stage_rows = []
for p in sorted(STAGES_DIR.glob("stage*_*.json")):
    if p.name == "stage4_winner.json":
        continue
    d = json.loads(p.read_text())
    if d.get("protocol_version") != PROTOCOL_VERSION:
        continue
    stage_rows.append({
        "stage": d["stage"],
        "encoder": d["encoder"],
        "mode": d["mode"],
        "baseline_val_mae": d["baseline_val_mae_db"],
        "tuned_val_mae": d["best_val_mae_db"],
        "delta_mae_db": d["delta_mae_db"],
        "best_tuned_params": json.dumps(d["best_tuned_params"], sort_keys=True),
        "best_full_params": json.dumps(d["best_full_params"], sort_keys=True),
    })

df_stages = pd.DataFrame(stage_rows)
if not df_stages.empty:
    df_stages = df_stages.sort_values(["stage", "tuned_val_mae"]).reset_index(drop=True)
    df_stages.to_csv(REPORT_DIR / "table3_stage_winners_val.csv", index=False)
    print("\n=== TABEL 3: RINGKASAN STAGE 1-4 (BASELINE vs TUNED, VALIDATION) ===")
    display(df_stages[["stage", "encoder", "mode", "baseline_val_mae", "tuned_val_mae", "delta_mae_db", "best_tuned_params"]])

# --- 4. Tabel Ablation Akhir Test (3 Seed, Mean +- Std) ----------------------
TEST_ORDER = [
    "test_ablation_unimodal_audio.json",
    "test_ablation_unimodal_text.json",
    "test_ablation_bimodal_text_only_inference.json",
    "test_ablation_bimodal_audio_only_inference.json",
    "test_ablation_bimodal_baseline.json",
    "test_ablation_bimodal_final.json",
]
extra_ctrl = sorted(p.name for p in TEST_DIR.glob("test_ablation_unimodal_audio_same_encoder_*.json"))

test_rows = []
for fname in TEST_ORDER + extra_ctrl:
    p = TEST_DIR / fname
    if not p.is_file():
        continue
    d = json.loads(p.read_text())
    if d.get("protocol_version") != PROTOCOL_VERSION:
        continue
    th = d.get("test_human", {})
    ts = d.get("test_synthetic", {})
    test_rows.append({
        "variant": d["name"],
        "encoder": d["encoder"],
        "mode": d["mode"],
        "n_seeds": d["n_seeds"],
        "test_human_mae_mean": th.get("mae_db_mean"),
        "test_human_mae_std": th.get("mae_db_std"),
        "test_human_rmse_mean": th.get("rmse_db_mean"),
        "test_human_band_mae": th.get("band_mae_db_mean"),
        "test_human_success_1db": th.get("success_rate_1db_mean"),
        "test_human_adherence": th.get("instruction_adherence_mean"),
        "test_synth_mae_mean": ts.get("mae_db_mean"),
        "test_synth_mae_std": ts.get("mae_db_std"),
        "test_synth_band_mae": ts.get("band_mae_db_mean"),
        "test_synth_adherence": ts.get("instruction_adherence_mean"),
        "params": json.dumps(d.get("params", {}), sort_keys=True),
    })

df_test = pd.DataFrame(test_rows)
if not df_test.empty:
    df_test.to_csv(REPORT_DIR / "table4_final_test_ablation.csv", index=False)
    print("\n=== TABEL 4: ABLATION AKHIR TEST (3 SEEDS, MEAN +- STD) ===")
    disp = df_test.copy()
    disp["test_human_MAE"] = disp.apply(
        lambda r: f"{r['test_human_mae_mean']:.4f} ± {r['test_human_mae_std']:.4f}" if pd.notna(r["test_human_mae_mean"]) else "-",
        axis=1,
    )
    disp["test_synth_MAE"] = disp.apply(
        lambda r: f"{r['test_synth_mae_mean']:.4f} ± {r['test_synth_mae_std']:.4f}" if pd.notna(r["test_synth_mae_mean"]) else "-",
        axis=1,
    )
    display(disp[["variant", "encoder", "n_seeds", "test_human_MAE", "test_synth_MAE", "test_human_band_mae", "test_human_adherence"]])
else:
    print("\nTabel 4 (Ablation Akhir Test): belum ada hasil Phase 3.")

In [ ]:
# =============================================================================
# CELL 9: VISUALISASI ANALISIS (FIGURES 180 DPI)
# =============================================================================
import matplotlib.pyplot as plt
from IPython.display import Image, display as ipy_display

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 10,
})

# --- Figure 1: Baseline vs Tuned Validation MAE across Encoders & Modes ------
if not df_stages.empty:
    s1_map = df_stages[df_stages["stage"] == "stage1_audio_unimodal"].set_index("encoder").to_dict("index")
    s3_map = df_stages[df_stages["stage"] == "stage3_bimodal_audio"].set_index("encoder").to_dict("index")
    s2_row = df_stages[df_stages["stage"] == "stage2_text_unimodal"]
    encs_present = [e for e in CONFIG["encoders"] if e in s1_map or e in s3_map]

    if encs_present:
        fig, ax = plt.subplots(figsize=(10, 4.8))
        x = np.arange(len(encs_present))
        w = 0.20

        series = [
            ("Audio-Only Baseline", [s1_map[e]["baseline_val_mae"] if e in s1_map else np.nan for e in encs_present], "#94a3b8"),
            ("Audio-Only Tuned (S1)", [s1_map[e]["tuned_val_mae"] if e in s1_map else np.nan for e in encs_present], "#3b82f6"),
            ("Bimodal Baseline", [s3_map[e]["baseline_val_mae"] if e in s3_map else np.nan for e in encs_present], "#fbbf24"),
            ("Bimodal Audio-Tuned (S3)", [s3_map[e]["tuned_val_mae"] if e in s3_map else np.nan for e in encs_present], "#10b981"),
        ]
        for idx, (label, vals, color) in enumerate(series):
            bars = ax.bar(x + (idx - 1.5) * w, vals, width=w, label=label, color=color, edgecolor="white")
            for b, v in zip(bars, vals):
                if np.isfinite(v):
                    ax.text(b.get_x() + b.get_width() / 2, v + 0.03, f"{v:.2f}", ha="center", va="bottom", fontsize=8)

        if not s2_row.empty:
            t_val = float(s2_row.iloc[0]["tuned_val_mae"])
            ax.axhline(t_val, color="#ef4444", linestyle="--", linewidth=1.4, label=f"Text-Only Tuned S2 ({t_val:.2f} dB)")

        ax.set_xticks(x)
        ax.set_xticklabels(encs_present)
        ax.set_ylabel("Validation MAE (dB) — lebih rendah lebih baik")
        ax.set_title("Perbandingan Validation MAE: Baseline vs Tuned (Stage 1–3)")
        ax.legend(frameon=False, ncol=2, loc="upper right")
        fig.tight_layout()
        fig1_path = FIG_DIR / "fig1_baseline_vs_tuned_val.png"
        fig.savefig(fig1_path, dpi=180)
        plt.close(fig)
        print("Saved:", fig1_path)
        ipy_display(Image(filename=str(fig1_path)))

# --- Figure 2: Sensitivitas Hyperparameter OFAT (Stage 1-4) ------------------
if not df_ofat.empty:
    stage_knob_pairs = list(df_ofat[["stage", "knob"]].drop_duplicates().itertuples(index=False, name=None))
    n_panels = len(stage_knob_pairs)
    if n_panels > 0:
        ncols = min(4, n_panels)
        nrows = int(np.ceil(n_panels / ncols))
        fig, axes = plt.subplots(nrows, ncols, figsize=(4.2 * ncols, 3.4 * nrows), squeeze=False)
        for i, (st, kn) in enumerate(stage_knob_pairs):
            ax = axes[i // ncols][i % ncols]
            sub = df_ofat[(df_ofat["stage"] == st) & (df_ofat["knob"] == kn)]
            for enc, grp in sub.groupby("encoder"):
                grp = grp.sort_values("value")
                ax.plot(grp["value"].astype(str), grp["val_mae_db"], marker="o", linewidth=2, label=enc)
            ax.set_title(f"{st.replace('_', ' ')}\nknob: {kn}", fontsize=9)
            ax.set_ylabel("Val MAE (dB)")
            ax.tick_params(axis="x", rotation=20)
            ax.legend(fontsize=8, frameon=False)
        for j in range(n_panels, nrows * ncols):
            axes[j // ncols][j % ncols].axis("off")
        fig.suptitle("Kurva Sensitivitas OFAT Hyperparameter Tuning", y=1.02, fontsize=12)
        fig.tight_layout()
        fig2_path = FIG_DIR / "fig2_ofat_sensitivity.png"
        fig.savefig(fig2_path, dpi=180, bbox_inches="tight")
        plt.close(fig)
        print("Saved:", fig2_path)
        ipy_display(Image(filename=str(fig2_path)))

# --- Figure 3: Ablation Akhir Test (3 Seed, Mean +- Std) ---------------------
if not df_test.empty:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), sharey=True)
    y = np.arange(len(df_test))
    colors = ["#64748b", "#f97316", "#eab308", "#a855f7", "#3b82f6", "#10b981"][:len(df_test)]
    if len(colors) < len(df_test):
        colors += ["#0ea5e9"] * (len(df_test) - len(colors))

    for ax, mean_col, std_col, title in (
        (axes[0], "test_human_mae_mean", "test_human_mae_std", "Test Human (200 Prompt Manusia)"),
        (axes[1], "test_synth_mae_mean", "test_synth_mae_std", "Test Synthetic (2.000 Sampel)"),
    ):
        means = df_test[mean_col].astype(float).values
        stds = df_test[std_col].fillna(0.0).astype(float).values
        bars = ax.barh(y, means, xerr=stds, capsize=4, color=colors, edgecolor="white")
        for b, m, s in zip(bars, means, stds):
            if np.isfinite(m):
                ax.text(m + s + 0.05, b.get_y() + b.get_height() / 2, f"{m:.3f} ± {s:.3f}", va="center", fontsize=8.5)
        ax.set_xlabel("Test MAE (dB) — lebih rendah lebih baik")
        ax.set_title(title)

    axes[0].set_yticks(y)
    axes[0].set_yticklabels(df_test["variant"])
    axes[0].invert_yaxis()
    fig.suptitle("Ablation Akhir Test (3 Seeds, Mean ± Std)", fontsize=12)
    fig.tight_layout()
    fig3_path = FIG_DIR / "fig3_final_test_ablation.png"
    fig.savefig(fig3_path, dpi=180, bbox_inches="tight")
    plt.close(fig)
    print("Saved:", fig3_path)
    ipy_display(Image(filename=str(fig3_path)))

# --- Figure 4: Per-Stem MAE (Test Human) & Kurva Konvergensi 5 Epoch ---------
final_test_json = TEST_DIR / "test_ablation_bimodal_final.json"
cache_files = sorted(CACHE_DIR.glob("*.json"))
if final_test_json.is_file() or cache_files:
    fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.4))

    # Panel kiri: Per-stem MAE pada test_human untuk semua varian test yang ada
    ax = axes[0]
    stem_plotted = False
    x_s = np.arange(len(STEM_NAMES))
    avail_test_files = [TEST_DIR / f for f in TEST_ORDER if (TEST_DIR / f).is_file()]
    if avail_test_files:
        w = 0.8 / max(len(avail_test_files), 1)
        for i, tp in enumerate(avail_test_files):
            d = json.loads(tp.read_text())
            per_stem = d.get("test_human", {}).get("mae_per_stem_db", {})
            if not per_stem:
                continue
            vals = [per_stem[s]["mean"] for s in STEM_NAMES]
            errs = [per_stem[s]["std"] for s in STEM_NAMES]
            ax.bar(x_s + (i - (len(avail_test_files) - 1) / 2) * w, vals, width=w, yerr=errs, capsize=2, label=d["name"])
            stem_plotted = True
    if stem_plotted:
        ax.set_xticks(x_s)
        ax.set_xticklabels(STEM_NAMES)
        ax.set_ylabel("Test Human MAE (dB)")
        ax.set_title("Breakdown Error Per-Stem (Test Human)")
        ax.legend(fontsize=7.5, frameon=False)
    else:
        ax.text(0.5, 0.5, "Menunggu hasil Phase 3", ha="center", va="center", transform=ax.transAxes)
        ax.set_title("Breakdown Error Per-Stem")

    # Panel kanan: Kurva konvergensi 5 epoch dari baseline bimodal & text_only
    ax2 = axes[1]
    curve_plotted = False
    for cf in cache_files:
        d = json.loads(cf.read_text())
        if d.get("protocol_version") != PROTOCOL_VERSION or int(d.get("seed", -1)) != int(CONFIG["tuning_seed"]):
            continue
        hist = d.get("history", [])
        if not hist:
            continue
        # Tampilkan kurva run baseline (default params) untuk tiap encoder bimodal + text_only
        hp = d.get("cache_key_payload", {}).get("all_hyperparams", {})
        is_base_bimodal = (
            d["mode"] == "bimodal"
            and hp.get("wd_audio") == CONFIG["wd_audio"]
            and hp.get("lr_text") == CONFIG["lr_text"]
            and hp.get("text_hidden") == CONFIG["text_hidden"]
            and hp.get("lr_audio") == (CONFIG["lr_scratch"] if d["encoder"] == "cnn_scratch" else CONFIG["lr_audio"])
        )
        is_base_text = (
            d["mode"] == "text_only"
            and hp.get("lr_text") == CONFIG["lr_text"]
            and hp.get("text_hidden") == CONFIG["text_hidden"]
        )
        if is_base_bimodal or is_base_text:
            eps = [h["epoch"] for h in hist]
            maes = [h["val_mae_db"] for h in hist]
            lbl = "text_only (base)" if is_base_text else f"{d['encoder']} bimodal (base)"
            ax2.plot(eps, maes, marker="o", linewidth=2, label=lbl)
            curve_plotted = True
    if curve_plotted:
        ax2.set_xticks(range(1, MAX_EPOCHS + 1))
        ax2.set_xlabel("Epoch (Max 5)")
        ax2.set_ylabel("Validation MAE (dB)")
        ax2.set_title("Kurva Konvergensi 5 Epoch (Baseline)")
        ax2.legend(fontsize=8, frameon=False)
    else:
        ax2.text(0.5, 0.5, "Belum ada history kurva baseline", ha="center", va="center", transform=ax2.transAxes)
        ax2.set_title("Kurva Konvergensi 5 Epoch")

    fig.tight_layout()
    fig4_path = FIG_DIR / "fig4_per_stem_and_convergence.png"
    fig.savefig(fig4_path, dpi=180)
    plt.close(fig)
    print("Saved:", fig4_path)
    ipy_display(Image(filename=str(fig4_path)))

In [ ]:
# =============================================================================
# CELL 10: PARITY, ANTI-LEAKAGE, & PROTOCOL SANITY CHECKS
# =============================================================================
# 1. Verifikasi holdout split 80/20/50 bebas song overlap & prompt overlap
tr = load_manifest_records(Path(CONFIG["train_manifest"]))
va = load_manifest_records(Path(CONFIG["val_manifest"]))
th = load_manifest_records(Path(CONFIG["test_human_manifest"]))
ts = load_manifest_records(Path(CONFIG["test_synthetic_manifest"]))

tr_songs, va_songs = {r["song"] for r in tr}, {r["song"] for r in va}
te_songs = {r["song"] for r in th} | {r["song"] for r in ts}
assert len(tr_songs) == 80 and len(va_songs) == 20 and len(te_songs) == 50
assert not (tr_songs & va_songs) and not (tr_songs & te_songs) and not (va_songs & te_songs), "Song overlap terdeteksi!"
assert len(tr) == 3200 and len(va) == 800 and len(th) == 200 and len(ts) == 2000

tr_p, va_p = {r["prompt"] for r in tr}, {r["prompt"] for r in va}
te_p = {r["prompt"] for r in th} | {r["prompt"] for r in ts}
assert not (tr_p & va_p) and not (tr_p & te_p) and not (va_p & te_p), "Prompt overlap terdeteksi!"

# 2. Verifikasi nilai default CONFIG ada di dalam search space (jaminan deduplikasi cache)
assert CONFIG["lr_audio"] in AUDIO_SEARCH_SPACE["pretrained"]["lr_audio"]
assert CONFIG["wd_audio"] in AUDIO_SEARCH_SPACE["pretrained"]["wd_audio"]
assert CONFIG["lr_scratch"] in AUDIO_SEARCH_SPACE["scratch"]["lr_scratch"]
assert CONFIG["wd_audio"] in AUDIO_SEARCH_SPACE["scratch"]["wd_audio"]
assert CONFIG["lr_text"] in TEXT_SEARCH_SPACE["lr_text"]
assert CONFIG["text_hidden"] in TEXT_SEARCH_SPACE["text_hidden"]

# 3. Verifikasi protokol epoch = 5 & 3 seed final di seluruh file cache dan test
assert CONFIG["epochs"] == MAX_EPOCHS == 5
assert len(CONFIG["final_seeds"]) == 3
for cf in CACHE_DIR.glob("*.json"):
    d = json.loads(cf.read_text())
    assert d.get("protocol_version") == PROTOCOL_VERSION, f"Protokol tidak cocok: {cf.name}"
    assert int(d["cache_key_payload"]["max_epochs"]) == 5, f"Bukan 5 epoch: {cf.name}"
    assert int(d.get("epochs_run", 0)) <= 5, f"Melebihi 5 epoch: {cf.name}"

for tf in TEST_DIR.glob("test_ablation_*.json"):
    d = json.loads(tf.read_text())
    assert int(d["n_seeds"]) == 3, f"Test ablation wajib 3 seed: {tf.name}"

# 4. Verifikasi keunikan cache key lintas mode, seed, dan parameter terkunci
_k_bi = cache_key_hash(canonical_cache_payload("ast", "bimodal", 42, make_config(encoder_name="ast", mode="bimodal")))
_k_au = cache_key_hash(canonical_cache_payload("ast", "audio_only", 42, make_config(encoder_name="ast", mode="audio_only")))
_k_tx = cache_key_hash(canonical_cache_payload("ast", "text_only", 42, make_config(encoder_name="ast", mode="text_only")))
_k_s2 = cache_key_hash(canonical_cache_payload("ast", "bimodal", 123, make_config(encoder_name="ast", mode="bimodal")))
_k_dr = cache_key_hash(canonical_cache_payload("ast", "bimodal", 42, make_config({"dropout_audio": 0.25}, "ast", "bimodal")))
assert len({_k_bi, _k_au, _k_tx, _k_s2, _k_dr}) == 5, "Tabrakan cache key terdeteksi!"

print("SEMUA PARITY CHECK LOLOS:")
print(" - Holdout 80/20/50 bebas song overlap & prompt overlap")
print(" - Semua nilai baseline berada dalam search space OFAT")
print(" - Semua hasil terverifikasi max 5 epoch & protocol_version:", PROTOCOL_VERSION)
print(" - Strict cache key bebas tabrakan antar mode, seed, dan locked hyperparams")

In [ ]:
# =============================================================================
# CELL 11: PACKAGING OUTPUT AGAR MUDAH DIUNDUH (KAGGLE & COLAB)
# =============================================================================
sync_to_drive(include_checkpoints=EXPORT_FINAL_CHECKPOINTS)

EXPORT_ROOT = WORK_ROOT  # /kaggle/working (Kaggle) atau /content (Colab)
results_zip_path = EXPORT_ROOT / f"wadidaw_results_member{MEMBER}_{PHASE}.zip"
ckpts_zip_path = EXPORT_ROOT / f"wadidaw_checkpoints_member{MEMBER}.zip"

# 1. ZIP ringan: semua JSON cache, baseline, stages, test_ablation, CSV reports, dan PNG figures
#    (tanpa file .pt agar kecil & cepat diunduh/dibagikan antar anggota)
LIGHT_SUBDIRS = ("run_cache", "baseline", "stages", "test_ablation", "reports", "figures")
light_files = [
    p for sub in LIGHT_SUBDIRS if (OUT_DIR / sub).is_dir()
    for p in sorted((OUT_DIR / sub).rglob("*"))
    if p.is_file() and p.suffix != ".pt"
]

if light_files:
    with zipfile.ZipFile(results_zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for fpath in light_files:
            zf.write(fpath, arcname=str(fpath.relative_to(OUT_DIR)))
    print(f"[ZIP HASIL RINGAN] {results_zip_path} ({results_zip_path.stat().st_size / 1e6:.2f} MB, {len(light_files)} file)")
    if DRIVE_AVAILABLE:
        shutil.copy2(results_zip_path, DRIVE_OUTPUT_DIR / results_zip_path.name)
        print(f"  -> tersalin juga ke Google Drive: {DRIVE_OUTPUT_DIR / results_zip_path.name}")
else:
    print("Belum ada file hasil untuk di-zip.")

# 2. ZIP terpisah untuk checkpoint .pt (hanya bila ada checkpoint di sesi ini & EXPORT_FINAL_CHECKPOINTS=True)
ckpt_files = sorted(CKPT_DIR.glob("*.pt")) if EXPORT_FINAL_CHECKPOINTS else []
if ckpt_files:
    with zipfile.ZipFile(ckpts_zip_path, "w", compression=zipfile.ZIP_STORED) as zf:
        for fpath in ckpt_files:
            zf.write(fpath, arcname=f"checkpoints/{fpath.name}")
    print(f"[ZIP CHECKPOINTS]  {ckpts_zip_path} ({ckpts_zip_path.stat().st_size / 1e6:.2f} MB, {len(ckpt_files)} file)")

# 3. Kemudahan download di Kaggle & Colab
if IN_KAGGLE:
    print("\nDi Kaggle: buka panel kanan -> Output -> klik tombol Download pada:")
    print(f" - {results_zip_path.name}")
    if ckpt_files:
        print(f" - {ckpts_zip_path.name} (opsional, untuk inference/deploy)")
elif IN_COLAB:
    if AUTO_TRIGGER_COLAB_DOWNLOAD and results_zip_path.is_file():
        from google.colab import files
        files.download(str(results_zip_path))
    else:
        print("\nDi Colab: unduh dari panel Files sebelah kiri (atau langsung ambil dari Google Drive):")
        print(f" - Lokal VM: {results_zip_path}")
        if DRIVE_AVAILABLE:
            print(f" - Google Drive: {DRIVE_OUTPUT_DIR}")